# Adaptive Shield Weekly Report (MVP)

"
"This notebook automates weekly reporting for **configuration drift** and **integration failure** alerts.

"
"## Pipeline
"
"1. Fetch alerts in lookback window
"
"2. Filter target alert types
"
"3. Enrich with security check details
"
"4. Enrich affected entities
"
"5. Build summary + entities tables
"
"6. Export XLSX + CSV

"
"ServiceNow enrichment can be enabled with Selenium-based incident mapping.


In [ ]:
# Cell 1: Standalone Initialization (bootstrap + functions + variables)
from __future__ import annotations

import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "requests": "requests>=2.31.0",
    "pandas": "pandas>=2.0.0",
    "dotenv": "python-dotenv>=1.0.0",
    "openpyxl": "openpyxl>=3.1.2",
    "selenium": "selenium>=4.15.0",
    "webdriver_manager": "webdriver-manager>=4.0.0",
    "ipywidgets": "ipywidgets>=8.0.0",
}

missing_specs = []
for module_name, package_spec in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        missing_specs.append(package_spec)

if missing_specs:
    print("Installing missing dependencies:", ", ".join(missing_specs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_specs])
    importlib.invalidate_caches()
else:
    print("All required dependencies are already installed.")

failed_modules = []
for module_name, package_spec in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        failed_modules.append(f"{package_spec}: {exc}")

if failed_modules:
    raise RuntimeError(
        "Dependency bootstrap failed. Missing modules after install: "
        + " | ".join(failed_modules)
    )

print("Dependency bootstrap completed successfully.")

def _extract_incident_last_note(
    driver: Any,
    incident_url: str,
    timeout_seconds: int,
    *,
    cache_store: WebCacheStore | None = None,
    cache_flags: dict[str, Any] | None = None,
) -> tuple[str, str, str]:
    from selenium.webdriver.common.by import By

    if not incident_url:
        return "", "", ""

    cache_key = _cache_key("snow_detail", incident_url)
    if cache_store and _is_cache_enabled(cache_flags, "snow_detail"):
        cached = cache_store.get("snow_detail", cache_key)
        if isinstance(cached, dict):
            return (
                _safe_text(cached.get("note_text")),
                _safe_text(cached.get("note_source")),
                _safe_text(cached.get("updated_datetime")),
            )

    note_text = ""
    note_source = ""
    updated_datetime = ""
    original_url = driver.current_url

    try:
        driver.get(incident_url)
        try:
            _wait_for_document_ready(driver, timeout_seconds)
        except Exception:
            pass

        for selector in (
            "input#incident\\.sys_updated_on",
            "input[id*='sys_updated_on']",
            "input[name='sys_updated_on']",
        ):
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            if elements:
                updated_datetime = _element_text_or_value(elements[0])
                if updated_datetime:
                    break

        selector_pairs = [
            ("work_note", "textarea#activity-stream-work_notes-textarea"),
            ("comment", "textarea#activity-stream-comments-textarea"),
            ("work_note", "textarea[id*='work_notes']"),
            ("comment", "textarea[id*='comments']"),
            ("resolution_note", "textarea#incident\\.close_notes"),
            ("resolution_note", "textarea[id*='close_notes']"),
            ("resolution_note", "input[id*='close_notes']"),
        ]

        for source_name, selector in selector_pairs:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            for element in elements:
                candidate = _element_text_or_value(element)
                if candidate:
                    note_text = candidate
                    note_source = source_name
                    break
            if note_text:
                break

        if not note_text:
            stream_elements = driver.find_elements(By.CSS_SELECTOR, "div.activity-stream-text, div.sn-stream-input-decorator")
            for element in stream_elements:
                candidate = _element_text_or_value(element)
                if candidate:
                    note_text = candidate
                    note_source = "activity_stream"
                    break

        if cache_store and _is_cache_enabled(cache_flags, "snow_html"):
            try:
                cache_store.set_html(
                    "snow_html",
                    _cache_key("snow_detail_html", incident_url),
                    driver.page_source,
                    metadata={"incident_url": incident_url, "stage": "detail"},
                )
            except Exception:
                pass

    except Exception:
        pass
    finally:
        try:
            driver.get(original_url)
            try:
                _wait_for_document_ready(driver, min(timeout_seconds, 15))
            except Exception:
                pass
        except Exception:
            pass

    if cache_store and _is_cache_enabled(cache_flags, "snow_detail"):
        try:
            cache_store.set(
                "snow_detail",
                cache_key,
                {
                    "incident_url": incident_url,
                    "note_text": note_text,
                    "note_source": note_source,
                    "updated_datetime": updated_datetime,
                },
                metadata={"stage": "detail"},
            )
        except Exception:
            pass

    return note_text, note_source, updated_datetime

# Initialization helpers and runtime configuration
import os
import re
import json
import math
import gzip
import hashlib
import pickle
import time
import logging
from collections import defaultdict, deque
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Callable, Deque, Iterable
from urllib.parse import parse_qs, quote, urlparse

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import HTML, display, clear_output

try:
    import ipywidgets as widgets
except Exception:
    widgets = None


SUMMARY_COLUMNS = [
    "change_datetime",
    "security_check_name",
    "security_check_details",
    "remediation_steps",
    "impact_level",
    "current_status",
    "affected_entities_count",
    "affected_scope",
    "affected_entities_detail",
    "account_id",
    "account_name",
    "integration_id",
    "integration_name",
    "integration_alias",
    "saas_name",
    "security_check_id",
    "security_check_api_link",
    "alert_id",
    "alert_type",
    "source",
    "source_id",
    "is_archived",
    "ticket_number",
    "ticket_opened_datetime",
    "ticket_state",
    "ticket_assigned_to",
    "ticket_owner",
    "ticket_status",
    "ticket_last_update_datetime",
    "ticket_last_update_days_ago",
    "ticket_last_note_source",
    "ticket_last_update_content",
    "ticket_is_closed",
    "ticket_match_key",
    "ticket_count_for_check",
    "open_ticket_count_for_check",
    "extracted_at_utc",
]

ENTITY_COLUMNS = [
    "account_id",
    "security_check_id",
    "alert_id",
    "entity_type",
    "entity_name",
    "entity_dismissed",
    "entity_dismissed_reason",
    "entity_dismiss_expiration_date",
    "entity_extra_context_json",
    "entity_usage_json",
    "entity_raw_json",
]

SNOW_COLUMNS = [
    "ticket_number",
    "ticket_opened_datetime",
    "ticket_state",
    "ticket_assigned_to",
    "ticket_owner",
    "ticket_status",
    "ticket_last_update_datetime",
    "ticket_last_update_days_ago",
    "ticket_last_note_source",
    "ticket_last_update_content",
    "ticket_is_closed",
    "ticket_match_key",
    "ticket_count_for_check",
    "open_ticket_count_for_check",
]

SNOW_RAW_COLUMNS = [
    "ticket_number",
    "short_description",
    "ticket_opened_datetime",
    "ticket_state",
    "ticket_assigned_to",
    "ticket_last_update_datetime",
    "ticket_last_update_days_ago",
    "ticket_last_note_source",
    "ticket_last_update_content",
    "ticket_is_closed",
    "ticket_match_key",
    "match_mode",
    "incident_url",
    "raw_opened_value",
    "raw_updated_value",
]

SNOW_DETAIL_COLUMNS = [
    "ticket_match_key",
    "ticket_number",
    "ticket_url",
    "ticket_opened_datetime",
    "ticket_state",
    "ticket_assigned_to",
    "ticket_owner",
    "ticket_status",
    "ticket_last_update_datetime",
    "ticket_last_update_days_ago",
    "ticket_last_note_source",
    "ticket_last_update_content",
    "ticket_is_closed",
    "ticket_count_for_check",
    "open_ticket_count_for_check",
]

JOIN_KEY = "alert_id"
TARGET_ALERT_TYPES = {"configuration_drift", "integration_failure"}
_TYPE_ALIAS_MAP = {
    "security_check_degraded": "check_degraded",
    "security_check_degrade": "check_degraded",
    "securitycheckdegraded": "check_degraded",
    "threat": "threat",
}


AS_CACHE_DATASET_KEYS = (
    "as_accounts",
    "as_alerts",
    "as_security_checks",
    "as_affected_entities",
)

SNOW_CACHE_DATASET_KEYS = (
    "snow_list",
    "snow_detail",
    "snow_html",
)

CACHE_DATASET_KEYS = (*AS_CACHE_DATASET_KEYS, *SNOW_CACHE_DATASET_KEYS)

CACHE_FLAG_DEFAULTS = {key: True for key in CACHE_DATASET_KEYS}


def _build_runtime_cache_flags(
    *,
    use_cache_adaptive_shield: bool,
    use_cache_service_now: bool,
) -> dict[str, bool]:
    flags: dict[str, bool] = {}
    for key in AS_CACHE_DATASET_KEYS:
        flags[key] = bool(use_cache_adaptive_shield)
    for key in SNOW_CACHE_DATASET_KEYS:
        flags[key] = bool(use_cache_service_now)
    return flags


def _cache_group_enabled(
    runtime_cache_flags: dict[str, bool],
    dataset_keys: tuple[str, ...],
    *,
    default: bool,
) -> bool:
    if not dataset_keys:
        return default
    return all(bool(runtime_cache_flags.get(key, default)) for key in dataset_keys)

def _cache_key(*parts: Any) -> str:
    return json.dumps(parts, sort_keys=True, ensure_ascii=True, default=str, separators=(",", ":"))


def _is_cache_enabled(cache_flags: dict[str, Any] | None, dataset: str) -> bool:
    if dataset not in CACHE_FLAG_DEFAULTS:
        return False
    if not isinstance(cache_flags, dict):
        return CACHE_FLAG_DEFAULTS[dataset]
    return bool(cache_flags.get(dataset, CACHE_FLAG_DEFAULTS[dataset]))


class WebCacheStore:
    def __init__(self, root_dir: Path | str, run_day_dir: Path | str | None = None) -> None:
        self.root_dir = Path(root_dir).expanduser().resolve()
        self.root_dir.mkdir(parents=True, exist_ok=True)
        self.run_day_dir = Path(run_day_dir).expanduser().resolve() if run_day_dir else None
        if self.run_day_dir is not None:
            (self.run_day_dir / "cache_dump").mkdir(parents=True, exist_ok=True)

    def _hash(self, dataset: str, cache_key: str) -> str:
        digest_src = f"{dataset}::{cache_key}".encode("utf-8", errors="ignore")
        return hashlib.sha1(digest_src).hexdigest()

    def _dataset_dir(self, dataset: str) -> Path:
        path = self.root_dir / dataset
        path.mkdir(parents=True, exist_ok=True)
        return path

    def _json_path(self, dataset: str, cache_key: str) -> Path:
        return self._dataset_dir(dataset) / f"{self._hash(dataset, cache_key)}.json"

    def _html_path(self, dataset: str, cache_key: str) -> Path:
        return self._dataset_dir(dataset) / f"{self._hash(dataset, cache_key)}.html.gz"

    def get(self, dataset: str, cache_key: str) -> Any | None:
        path = self._json_path(dataset, cache_key)
        if not path.exists():
            return None
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return None
        if isinstance(payload, dict) and "payload" in payload:
            return payload.get("payload")
        return payload

    def set(self, dataset: str, cache_key: str, payload: Any, *, metadata: dict[str, Any] | None = None) -> None:
        path = self._json_path(dataset, cache_key)
        document = {
            "dataset": dataset,
            "cache_key": cache_key,
            "saved_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "metadata": metadata or {},
            "payload": payload,
        }
        path.write_text(json.dumps(document, ensure_ascii=False, default=str), encoding="utf-8")
        self.snapshot(
            dataset,
            {
                "cache_key": cache_key,
                "cache_path": str(path),
                "metadata": metadata or {},
            },
        )

    def set_html(self, dataset: str, cache_key: str, html_text: str, *, metadata: dict[str, Any] | None = None) -> str:
        path = self._html_path(dataset, cache_key)
        with gzip.open(path, "wt", encoding="utf-8") as fh:
            fh.write(html_text or "")
        self.snapshot(
            dataset,
            {
                "cache_key": cache_key,
                "html_gz_path": str(path),
                "html_bytes": len((html_text or "").encode("utf-8", errors="ignore")),
                "metadata": metadata or {},
            },
        )
        return str(path)

    def snapshot(self, dataset: str, record: dict[str, Any]) -> None:
        if self.run_day_dir is None:
            return
        snapshot_path = self.run_day_dir / "cache_dump" / f"{dataset}.jsonl"
        row = {
            "dataset": dataset,
            "snapshot_ts_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            **record,
        }
        with snapshot_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


class AdaptiveShieldClientError(RuntimeError):
    """Base exception for Adaptive Shield client failures."""


class AuthenticationError(AdaptiveShieldClientError):
    """Raised when API key is invalid or missing required permissions."""


class ApiRequestError(AdaptiveShieldClientError):
    """Raised when request retries are exhausted."""


class AdaptiveShieldClient:
    """Client for Adaptive Shield REST API."""

    def __init__(
        self,
        api_key: str,
        base_url: str = "https://api.adaptive-shield.com",
        *,
        rate_limit_per_minute: int = 90,
        timeout_seconds: int = 30,
        max_retries: int = 3,
        session: requests.Session | None = None,
        sleep_func: Callable[[float], None] = time.sleep,
        time_func: Callable[[], float] = time.monotonic,
        cache_store: WebCacheStore | None = None,
        cache_flags: dict[str, Any] | None = None,
    ) -> None:
        if not api_key:
            raise ValueError("AS_API_KEY is required")

        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds
        self.max_retries = max_retries
        self.rate_limit_per_minute = rate_limit_per_minute
        self._sleep = sleep_func
        self._time = time_func
        self._request_times: Deque[float] = deque()
        self._cache_store = cache_store
        self._cache_flags = cache_flags if isinstance(cache_flags, dict) else {}
        self._session = session or requests.Session()
        self._session.headers.update(
            {
                "Authorization": f"Token {api_key}",
                "Content-Type": "application/json",
            }
        )

    def get_accounts(self) -> list[dict[str, Any]]:
        records = self._paginate("GET", "/api/v1/accounts")
        return [item for item in records if isinstance(item, dict)]

    def get_alerts(
        self,
        account_id: str,
        from_date: str,
        to_date: str,
        alert_type: str | None = None,
    ) -> list[dict[str, Any]]:
        path = f"/api/v1/accounts/{account_id}/alerts"
        params: dict[str, Any] = {"from_date": from_date, "to_date": to_date}
        if alert_type:
            params["type"] = alert_type
        records = self._paginate("GET", path, params=params)
        return [item for item in records if isinstance(item, dict)]

    def get_security_check(self, account_id: str, security_check_id: str) -> dict[str, Any]:
        path = f"/api/v1/accounts/{account_id}/security_checks/{security_check_id}"
        payload = self._request("GET", path)
        if isinstance(payload, dict) and isinstance(payload.get("data"), dict):
            return payload["data"]
        if isinstance(payload, dict):
            return payload
        return {}

    def get_affected_entities(
        self,
        account_id: str,
        security_check_id: str,
        *,
        limit: int = 100,
        offset: int = 0,
    ) -> list[dict[str, Any]]:
        path = f"/api/v1/accounts/{account_id}/security_checks/{security_check_id}/affected"
        params: dict[str, Any] = {"limit": limit, "offset": offset}
        records = self._paginate("GET", path, params=params)
        return [item for item in records if isinstance(item, dict)]

    def get_integrations(self, account_id: str) -> list[dict[str, Any]]:
        path = f"/api/v1/accounts/{account_id}/integrations"
        records = self._paginate("GET", path)
        return [item for item in records if isinstance(item, dict)]

    def _paginate(
        self,
        method: str,
        path_or_url: str,
        params: dict[str, Any] | None = None,
    ) -> list[Any]:
        all_items: list[Any] = []
        seen_ids: set[str] = set()
        seen_pages: set[tuple[str, tuple[tuple[str, str], ...]]] = set()

        original_path = path_or_url
        next_path = path_or_url
        next_params = dict(params or {})

        while True:
            page_key = (
                next_path,
                tuple(sorted((str(k), str(v)) for k, v in (next_params or {}).items())),
            )
            if page_key in seen_pages:
                logging.warning("Pagination loop detected for %s", next_path)
                break
            seen_pages.add(page_key)

            payload = self._request(method, next_path, params=next_params or None)
            items, next_page_uri, pagination = self._extract_page(payload)

            for item in items:
                if isinstance(item, dict) and item.get("id") is not None:
                    item_id = str(item["id"])
                    if item_id in seen_ids:
                        continue
                    seen_ids.add(item_id)
                all_items.append(item)

            if next_page_uri:
                next_path = next_page_uri
                next_params = {}
                continue

            next_from_meta = self._next_from_meta(
                original_path=original_path,
                current_params=next_params,
                pagination=pagination,
            )
            if next_from_meta is None:
                break

            next_path, next_params = next_from_meta

        return all_items

    def _request(
        self,
        method: str,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
    ) -> Any:
        url = self._resolve_url(path_or_url)
        cache_dataset = self._cache_dataset_for_request(url)
        cache_key = _cache_key(method.upper(), url, params or {})
        if cache_dataset and self._cache_store and _is_cache_enabled(self._cache_flags, cache_dataset):
            cached_payload = self._cache_store.get(cache_dataset, cache_key)
            if cached_payload is not None:
                logging.info("[CACHE][AS] hit dataset=%s", cache_dataset)
                return cached_payload
        attempt = 0

        while True:
            self._throttle()
            try:
                response = self._session.request(
                    method=method.upper(),
                    url=url,
                    params=params,
                    timeout=self.timeout_seconds,
                )
            except requests.RequestException as exc:
                if attempt >= self.max_retries:
                    raise ApiRequestError(
                        f"Request failed after retries: {method} {url}"
                    ) from exc
                wait_seconds = float(2 ** attempt)
                logging.warning(
                    "Request error on %s %s, retrying in %.1fs: %s",
                    method,
                    url,
                    wait_seconds,
                    exc,
                )
                self._sleep(wait_seconds)
                attempt += 1
                continue

            if response.status_code == 401:
                raise AuthenticationError(
                    "Unauthorized (401). Verify AS_API_KEY and account access."
                )

            if response.status_code == 429 and attempt < self.max_retries:
                retry_after = self._parse_retry_after(response.headers.get("Retry-After"))
                wait_seconds = retry_after if retry_after is not None else float(2 ** attempt)
                logging.warning(
                    "Rate limited on %s %s, retrying in %.1fs",
                    method,
                    url,
                    wait_seconds,
                )
                self._sleep(wait_seconds)
                attempt += 1
                continue

            if 500 <= response.status_code < 600 and attempt < self.max_retries:
                wait_seconds = float(2 ** attempt)
                logging.warning(
                    "Server error %s on %s %s, retrying in %.1fs",
                    response.status_code,
                    method,
                    url,
                    wait_seconds,
                )
                self._sleep(wait_seconds)
                attempt += 1
                continue

            if response.status_code >= 400:
                snippet = (response.text or "").strip()[:500]
                raise ApiRequestError(
                    f"HTTP {response.status_code} for {method} {url}: {snippet}"
                )

            try:
                payload = response.json()
            except ValueError as exc:
                raise ApiRequestError(f"Invalid JSON response for {method} {url}") from exc

            if cache_dataset and self._cache_store and _is_cache_enabled(self._cache_flags, cache_dataset):
                self._cache_store.set(
                    cache_dataset,
                    cache_key,
                    payload,
                    metadata={"method": method.upper(), "url": url, "params": params or {}},
                )
            return payload

    def _throttle(self) -> None:
        now = self._time()
        while self._request_times and now - self._request_times[0] >= 60:
            self._request_times.popleft()

        if len(self._request_times) >= self.rate_limit_per_minute:
            wait_seconds = 60 - (now - self._request_times[0]) + 0.01
            logging.info("Throttling API calls for %.2fs", wait_seconds)
            self._sleep(wait_seconds)
            now = self._time()
            while self._request_times and now - self._request_times[0] >= 60:
                self._request_times.popleft()

        self._request_times.append(now)

    def _resolve_url(self, path_or_url: str) -> str:
        if path_or_url.startswith("http://") or path_or_url.startswith("https://"):
            return path_or_url
        return f"{self.base_url}/{path_or_url.lstrip('/')}"

    @staticmethod
    def _cache_dataset_for_request(url: str) -> str | None:
        path = urlparse(url).path
        if path.endswith("/api/v1/accounts"):
            return "as_accounts"
        if "/alerts" in path:
            return "as_alerts"
        if "/security_checks/" in path and path.endswith("/affected"):
            return "as_affected_entities"
        if "/security_checks/" in path:
            return "as_security_checks"
        return None

    @staticmethod
    def _extract_page(payload: Any) -> tuple[list[Any], str | None, dict[str, Any] | None]:
        items: list[Any] = []
        next_page_uri: str | None = None
        pagination: dict[str, Any] | None = None

        if isinstance(payload, list):
            return payload, None, None

        if not isinstance(payload, dict):
            return [], None, None

        if isinstance(payload.get("data"), list):
            items = payload["data"]
        elif isinstance(payload.get("resources"), list):
            items = payload["resources"]
        elif isinstance(payload.get("data"), dict):
            data_obj = payload["data"]
            if isinstance(data_obj.get("result"), list):
                items = data_obj["result"]
        elif isinstance(payload.get("result"), list):
            items = payload["result"]

        if isinstance(payload.get("next_page_uri"), str):
            next_page_uri = payload["next_page_uri"]
        elif isinstance(payload.get("data"), dict) and isinstance(payload["data"].get("next_page_uri"), str):
            next_page_uri = payload["data"]["next_page_uri"]

        meta = payload.get("meta")
        if isinstance(meta, dict) and isinstance(meta.get("pagination"), dict):
            pagination = meta["pagination"]

        return items, next_page_uri, pagination

    @staticmethod
    def _parse_retry_after(header_value: str | None) -> float | None:
        if not header_value:
            return None
        try:
            return max(0.0, float(header_value))
        except (TypeError, ValueError):
            return None

    def _next_from_meta(
        self,
        *,
        original_path: str,
        current_params: dict[str, Any],
        pagination: dict[str, Any] | None,
    ) -> tuple[str, dict[str, Any]] | None:
        if not pagination:
            return None

        next_value = pagination.get("next")
        if isinstance(next_value, str):
            stripped = next_value.strip()
            if not stripped:
                return None
            if stripped.startswith("http://") or stripped.startswith("https://"):
                return stripped, {}
            if stripped.isdigit():
                params = dict(current_params)
                params["offset"] = int(stripped)
                return original_path, params
            parsed = urlparse(stripped)
            if parsed.query:
                params = dict(current_params)
                query_map = parse_qs(parsed.query)
                for key, values in query_map.items():
                    if values:
                        params[key] = values[-1]
                return (stripped if stripped.startswith("/") else original_path, params)

        if isinstance(next_value, (int, float)):
            params = dict(current_params)
            params["offset"] = int(next_value)
            return original_path, params

        if isinstance(next_value, dict):
            params = dict(current_params)
            for key in ("offset", "page", "cursor"):
                if key in next_value:
                    params[key] = next_value[key]
            return original_path, params

        if next_value in (None, "", False):
            offset = pagination.get("offset")
            limit = pagination.get("limit")
            total = pagination.get("total")
            if all(isinstance(v, int) for v in (offset, limit, total)):
                if offset + limit < total:
                    params = dict(current_params)
                    params["offset"] = offset + limit
                    params.setdefault("limit", limit)
                    return original_path, params

        return None


def normalize_alert_type(raw: Any) -> str:
    if raw is None:
        return ""

    value = str(raw).strip().lower()
    value = value.replace("-", "_").replace(" ", "_")
    value = re.sub(r"[^a-z0-9_]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return _TYPE_ALIAS_MAP.get(value, value)


def filter_target_alerts(
    rows: Iterable[dict[str, Any]],
    *,
    include_check_degraded: bool = False,
) -> list[dict[str, Any]]:
    targets = set(TARGET_ALERT_TYPES)
    if include_check_degraded:
        targets.add("check_degraded")

    filtered: list[dict[str, Any]] = []
    for row in rows:
        normalized = normalize_alert_type(row.get("alert_type") or row.get("type"))
        if bool(row.get("is_archived")):
            continue
        if normalized not in targets:
            continue

        item = dict(row)
        item["alert_type_normalized"] = normalized
        filtered.append(item)

    return filtered


def extract_security_check_id(alert: dict[str, Any]) -> str:
    for key in ("source_id", "security_check_id"):
        value = alert.get(key)
        if value:
            return str(value)

    api_link = alert.get("security_check_api_link")
    if not api_link:
        return ""

    match = re.search(r"/security_checks/([^/?#]+)", str(api_link))
    if match:
        return match.group(1)
    return ""


def _first_url_candidate(*values: Any) -> str:
    for raw in values:
        text = str(raw).strip() if raw is not None else ""
        if not text:
            continue
        if text.startswith(("http://", "https://", "/")):
            return text
    return ""


def _resolve_as_security_check_link(
    *,
    alert: dict[str, Any],
    check: dict[str, Any],
    account_id: str,
    security_check_id: str,
    as_base_url: str,
) -> str:
    candidate = _first_url_candidate(
        alert.get("security_check_url"),
        alert.get("security_check_link"),
        alert.get("security_check_api_link"),
        check.get("security_check_url"),
        check.get("security_check_link"),
        check.get("security_check_api_link"),
        check.get("url"),
        check.get("link"),
        check.get("api_link"),
    )
    if candidate:
        if candidate.startswith(("http://", "https://")):
            return candidate
        base = str(as_base_url or "").strip().rstrip("/")
        if base and candidate.startswith("/"):
            return f"{base}{candidate}"

    base = str(as_base_url or "").strip().rstrip("/")
    if not (base and account_id and security_check_id):
        return ""
    account_enc = quote(str(account_id), safe="")
    check_enc = quote(str(security_check_id), safe="")
    return f"{base}/api/v1/accounts/{account_enc}/security_checks/{check_enc}"


def _to_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"), default=str)


def _fallback_status_from_alert(alert: dict[str, Any]) -> str:
    normalized = normalize_alert_type(alert.get("alert_type") or alert.get("type"))
    if normalized == "integration_failure":
        return "Failed"
    if normalized == "configuration_drift":
        return "Drifted"
    if normalized == "check_degraded":
        return "Degraded"
    return ""


def build_entities_df(entity_records: Iterable[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for record in entity_records:
        account_id = record.get("account_id")
        alert_id = record.get("alert_id")
        security_check_id = record.get("security_check_id")
        entity = record.get("entity") if isinstance(record.get("entity"), dict) else record
        if not isinstance(entity, dict):
            continue

        rows.append(
            {
                "account_id": account_id,
                "security_check_id": security_check_id,
                "alert_id": alert_id,
                "entity_type": entity.get("type"),
                "entity_name": entity.get("entity_name") or entity.get("name"),
                "entity_dismissed": entity.get("dismissed"),
                "entity_dismissed_reason": entity.get("dismissed_reason"),
                "entity_dismiss_expiration_date": entity.get("dismiss_expiration_date"),
                "entity_extra_context_json": _to_json(entity.get("extra_context")),
                "entity_usage_json": _to_json(entity.get("usage")),
                "entity_raw_json": _to_json(entity),
            }
        )

    return pd.DataFrame(rows, columns=ENTITY_COLUMNS)


def build_summary_df(
    *,
    alerts: Iterable[dict[str, Any]],
    checks: Iterable[dict[str, Any]],
    entities: Iterable[dict[str, Any]],
    accounts: Iterable[dict[str, Any]] | None = None,
    extracted_at_utc: str | None = None,
    as_base_url: str = "https://api.adaptive-shield.com",
) -> pd.DataFrame:
    extracted_at = extracted_at_utc or datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    account_name_by_id: dict[str, str] = {}
    if accounts:
        for account in accounts:
            account_id = str(account.get("id") or account.get("account_id") or "")
            if account_id:
                account_name_by_id[account_id] = str(account.get("name") or account.get("account_name") or "")

    checks_by_alert: dict[str, dict[str, Any]] = {}
    checks_by_pair: dict[tuple[str, str], dict[str, Any]] = {}
    for check in checks:
        if not isinstance(check, dict):
            continue
        alert_id = str(check.get("alert_id") or "")
        account_id = str(check.get("account_id") or "")
        security_check_id = str(
            check.get("security_check_id")
            or check.get("id")
            or check.get("source_id")
            or ""
        )

        if alert_id:
            checks_by_alert[alert_id] = check
        if account_id and security_check_id:
            checks_by_pair[(account_id, security_check_id)] = check

    entity_names_by_alert: dict[str, list[str]] = defaultdict(list)
    entity_count_by_alert: dict[str, int] = defaultdict(int)
    for entity in entities:
        if not isinstance(entity, dict):
            continue
        alert_id = str(entity.get("alert_id") or "")
        if not alert_id:
            continue
        entity_name = entity.get("entity_name")
        if entity_name:
            entity_names_by_alert[alert_id].append(str(entity_name))
        entity_count_by_alert[alert_id] += 1

    rows: list[dict[str, Any]] = []
    for alert in alerts:
        if not isinstance(alert, dict):
            continue

        alert_id = str(alert.get("id") or alert.get("alert_id") or "")
        account_id = str(alert.get("account_id") or "")
        source_id = str(alert.get("source_id") or "")
        security_check_id = extract_security_check_id(alert) or source_id
        check = checks_by_alert.get(alert_id)
        if check is None and account_id and security_check_id:
            check = checks_by_pair.get((account_id, security_check_id), {})
        check = check or {}

        integration_obj = alert.get("integration") if isinstance(alert.get("integration"), dict) else {}
        if not integration_obj and isinstance(check.get("integration"), dict):
            integration_obj = check["integration"]

        is_global = bool(check.get("is_global"))
        entity_names = sorted(set(entity_names_by_alert.get(alert_id, [])))
        affected_diff = alert.get("affected_diff")
        affected_diff_names: list[str] = []
        if isinstance(affected_diff, list):
            affected_diff_names = [str(item) for item in affected_diff if item is not None]

        affected_scope = "global" if is_global else "unresolved"
        if not is_global:
            if entity_names:
                affected_scope = "entity"
            elif affected_diff_names:
                affected_scope = "entity_diff"

        if is_global:
            affected_entities_count: Any = "global"
        else:
            affected_entities_count = (
                check.get("affected")
                if check.get("affected") not in (None, "")
                else alert.get("new_affected_count")
            )
            if affected_entities_count in (None, "") and entity_count_by_alert.get(alert_id):
                affected_entities_count = entity_count_by_alert[alert_id]

        if is_global:
            affected_entities_detail = "global"
        elif entity_names:
            affected_entities_detail = "; ".join(entity_names)
        elif affected_diff_names:
            affected_entities_detail = "; ".join(affected_diff_names)
        else:
            affected_entities_detail = ""

        current_status = check.get("status") or _fallback_status_from_alert(alert)

        rows.append(
            {
                "change_datetime": alert.get("timestamp"),
                "security_check_name": check.get("name") or check.get("title"),
                "security_check_details": check.get("details") or alert.get("description"),
                "remediation_steps": check.get("remediation_plan"),
                "impact_level": check.get("impact"),
                "current_status": current_status,
                "affected_entities_count": affected_entities_count,
                "affected_scope": affected_scope,
                "affected_entities_detail": affected_entities_detail,
                "account_id": account_id,
                "account_name": account_name_by_id.get(account_id, ""),
                "integration_id": integration_obj.get("id") or check.get("integration_id") or check.get("Integration_id"),
                "integration_name": integration_obj.get("name") or check.get("integration_name") or check.get("saas_name"),
                "integration_alias": integration_obj.get("alias") or check.get("integration_alias"),
                "saas_name": check.get("saas_name") or integration_obj.get("name") or check.get("integration_name"),
                "security_check_id": security_check_id,
                "security_check_api_link": _resolve_as_security_check_link(
                    alert=alert,
                    check=check,
                    account_id=account_id,
                    security_check_id=security_check_id,
                    as_base_url=as_base_url,
                ),
                "alert_id": alert_id,
                "alert_type": normalize_alert_type(alert.get("alert_type") or alert.get("type")),
                "source": alert.get("source"),
                "source_id": source_id,
                "is_archived": bool(alert.get("is_archived")),
                "ticket_number": None,
                "ticket_opened_datetime": None,
                "ticket_state": None,
                "ticket_assigned_to": None,
                "ticket_owner": None,
                "ticket_status": None,
                "ticket_last_update_datetime": None,
                "ticket_last_update_days_ago": None,
                "ticket_last_note_source": None,
                "ticket_last_update_content": None,
                "ticket_is_closed": None,
                "ticket_match_key": None,
                "ticket_count_for_check": None,
                "open_ticket_count_for_check": None,
                "extracted_at_utc": extracted_at,
            }
        )

    return pd.DataFrame(rows, columns=SUMMARY_COLUMNS)


def _append_pipeline_error(
    pipeline_errors: list[dict[str, Any]] | None,
    *,
    stage: str,
    message: str,
    account_id: str | None = None,
    security_check_id: str | None = None,
    alert_id: str | None = None,
    ticket_number: str | None = None,
) -> None:
    if pipeline_errors is None:
        return
    pipeline_errors.append(
        {
            "stage": stage,
            "account_id": account_id,
            "security_check_id": security_check_id,
            "alert_id": alert_id,
            "ticket_number": ticket_number,
            "error": message,
            "extracted_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        }
    )


def normalize_match_text(raw: Any) -> str:
    if raw is None:
        return ""
    value = str(raw).strip().lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def build_check_key(saas_name: Any, integration_alias: Any, security_check_name: Any) -> str:
    saas_norm = normalize_match_text(saas_name)
    alias_norm = normalize_match_text(integration_alias)
    check_norm = normalize_match_text(security_check_name)
    if not check_norm:
        return ""
    return f"{saas_norm} | {alias_norm} | {check_norm}"


def parse_short_description_key(short_description: Any) -> str:
    text = str(short_description or "").strip()
    if not text:
        return ""
    if not re.search(r"adaptive\s*shield", text, flags=re.IGNORECASE):
        return ""

    split_parts = re.split(r"adaptive\s*shield", text, maxsplit=1, flags=re.IGNORECASE)
    suffix = split_parts[1] if len(split_parts) > 1 else text
    suffix = suffix.strip(" :-_	")

    if "|" in suffix:
        parts = [part.strip() for part in suffix.split("|")]
        if len(parts) >= 3:
            return build_check_key(parts[0], parts[1], parts[2])

    keyed = {
        "saas": "",
        "alias": "",
        "check": "",
    }
    for key_name in keyed:
        match = re.search(rf"{key_name}\s*[:=]\s*([^|,;]+)", suffix, flags=re.IGNORECASE)
        if match:
            keyed[key_name] = match.group(1).strip()

    if keyed["check"]:
        return build_check_key(keyed["saas"], keyed["alias"], keyed["check"])

    return ""


def _fallback_match_key(short_description: Any, candidate_keys: list[str]) -> str:
    desc_norm = normalize_match_text(short_description)
    if not desc_norm:
        return ""

    scored: list[tuple[int, str]] = []
    for key in candidate_keys:
        parts = [part.strip() for part in key.split("|") if part.strip()]
        if not parts:
            continue
        if all(part in desc_norm for part in parts):
            score = sum(len(part) for part in parts)
            scored.append((score, key))

    if not scored:
        return ""

    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[0][1]


def _parse_datetime_utc(raw: Any) -> datetime | None:
    if raw is None:
        return None
    if isinstance(raw, datetime):
        dt = raw
    else:
        text = str(raw).strip()
        if not text:
            return None

        dt = None
        parse_candidates = [text]
        if text.endswith("Z"):
            parse_candidates.append(text[:-1] + "+00:00")

        for candidate in parse_candidates:
            try:
                dt = datetime.fromisoformat(candidate)
                break
            except ValueError:
                continue

        if dt is None:
            formats = [
                "%Y-%m-%d %H:%M:%S",
                "%Y-%m-%d %H:%M",
                "%Y-%m-%d",
                "%m/%d/%Y %H:%M:%S",
                "%m/%d/%Y %H:%M",
                "%m/%d/%Y",
            ]
            for fmt in formats:
                try:
                    dt = datetime.strptime(text, fmt)
                    break
                except ValueError:
                    continue

        if dt is None:
            return None

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _to_iso_utc(raw: Any) -> str:
    dt = _parse_datetime_utc(raw)
    if dt is None:
        return ""
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def compute_days_ago(run_ts_utc: datetime, last_update_dt: Any) -> int | None:
    parsed = _parse_datetime_utc(last_update_dt)
    if parsed is None:
        return None
    delta = run_ts_utc - parsed
    return max(0, int(delta.total_seconds() // 86400))


def _is_closed_state(state_raw: Any) -> bool:
    if state_raw is None:
        return False

    if isinstance(state_raw, (int, float)):
        numeric = float(state_raw)
        if not math.isfinite(numeric):
            return False
        return int(numeric) in {6, 7, 8}

    text = normalize_match_text(state_raw)
    if text in {"6", "7", "8"}:
        return True

    closed_tokens = ("closed", "resolved", "complete", "completed", "cancelled", "canceled")
    return any(token in text for token in closed_tokens)


def _safe_text(raw: Any) -> str:
    if raw is None:
        return ""
    return str(raw).strip()


def _safe_int(raw: Any, default: int = 0) -> int:
    if raw in (None, ""):
        return default
    if isinstance(raw, bool):
        return int(raw)
    if isinstance(raw, (int, float)):
        numeric = float(raw)
        if not math.isfinite(numeric):
            return default
        return int(numeric)
    text = str(raw).strip()
    if not text:
        return default
    try:
        numeric = float(text)
        if not math.isfinite(numeric):
            return default
        return int(numeric)
    except ValueError:
        return default


def _element_text_or_value(element: Any) -> str:
    text = _safe_text(getattr(element, "text", ""))
    if text:
        return text
    for attr in ("value", "aria-label", "title"):
        attr_value = _safe_text(element.get_attribute(attr))
        if attr_value:
            return attr_value
    return ""


def _get_edge_session_profile(config: dict[str, Any]) -> str:
    """Copy auth cookies from the live Edge profile into a temp dir so
    Selenium can reuse the Microsoft SSO session without conflicting
    with the already-running Edge instance."""
    import platform
    import shutil
    import tempfile

    profile_name = _safe_text(config.get("snow_edge_profile", "Default")) or "Default"

    # Resolve base user-data-dir: explicit config > Windows default
    base = _safe_text(config.get("snow_user_data_dir"))
    if not base and platform.system() == "Windows":
        base = os.path.expandvars(r"%LOCALAPPDATA%\Microsoft\Edge\User Data")

    if not base or not os.path.isdir(base):
        return ""

    tmp = tempfile.mkdtemp(prefix="selenium_edge_")

    # Local State holds the DPAPI-wrapped master key used to decrypt cookies
    local_state_src = os.path.join(base, "Local State")
    if os.path.isfile(local_state_src):
        try:
            shutil.copy2(local_state_src, tmp)
        except Exception:
            pass

    # Copy just the Cookies file(s) from the chosen profile
    profile_src = os.path.join(base, profile_name)
    profile_dst = os.path.join(tmp, profile_name)
    os.makedirs(profile_dst, exist_ok=True)

    for rel in (os.path.join("Network", "Cookies"), "Cookies"):
        src_file = os.path.join(profile_src, rel)
        if os.path.isfile(src_file):
            dst_file = os.path.join(profile_dst, rel)
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)
            try:
                shutil.copy2(src_file, dst_file)
            except Exception:
                pass

    return tmp


def _build_snow_driver(config: dict[str, Any]) -> Any:
    from selenium import webdriver

    browser_raw = _safe_text(config.get("snow_browser", "edge")).lower()
    browser_aliases = {
        "msedge": "edge",
        "microsoft-edge": "edge",
        "microsoft_edge": "edge",
        "edgechromium": "edge",
        "auto": "edge",
        "any": "edge",
    }
    browser = browser_aliases.get(browser_raw, browser_raw)
    if browser not in {"edge", "chrome"}:
        logging.info("Unknown SNOW_BROWSER=%r, fallback to edge", browser_raw)
        browser = "edge"
    headless = config.get("snow_headless", True)
    driver_path = _safe_text(config.get("snow_chromedriver_path"))
    user_data_dir = _safe_text(config.get("snow_user_data_dir"))

    if browser == "edge":
        from selenium.webdriver.edge.options import Options
        from selenium.webdriver.edge.service import Service

        options = Options()
        if headless:
            options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--no-first-run")
        options.add_argument("--no-default-browser-check")

        # Reuse Microsoft SSO session from existing Edge profile
        session_dir = _get_edge_session_profile(config)
        if session_dir:
            options.add_argument(f"--user-data-dir={session_dir}")
            profile_name = _safe_text(config.get("snow_edge_profile", "Default")) or "Default"
            options.add_argument(f"--profile-directory={profile_name}")
            print(f"Edge session profile: {session_dir}")

        service = None
        if driver_path:
            service = Service(driver_path)
        else:
            try:
                from webdriver_manager.microsoft import EdgeChromiumDriverManager

                service = Service(EdgeChromiumDriverManager().install())
            except Exception:
                service = None

        if service is not None:
            return webdriver.Edge(service=service, options=options)
        return webdriver.Edge(options=options)

    else:  # default: chrome
        from selenium.webdriver.chrome.options import Options
        from selenium.webdriver.chrome.service import Service

        options = Options()
        if headless:
            options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        if user_data_dir:
            options.add_argument(f"--user-data-dir={user_data_dir}")

        service = None
        if driver_path:
            service = Service(driver_path)
        else:
            try:
                from webdriver_manager.chrome import ChromeDriverManager

                service = Service(ChromeDriverManager().install())
            except Exception:
                service = None

        if service is not None:
            return webdriver.Chrome(service=service, options=options)
        return webdriver.Chrome(options=options)


def _wait_for_document_ready(driver: Any, timeout_seconds: int) -> None:
    from selenium.webdriver.support.ui import WebDriverWait

    WebDriverWait(driver, timeout_seconds).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )


def _load_snow_session_cookies(driver: Any, *, base_url: str, cookie_path: Path) -> bool:
    if not cookie_path.exists():
        return False
    try:
        raw_cookies = pickle.loads(cookie_path.read_bytes())
    except Exception:
        return False

    if not isinstance(raw_cookies, list):
        return False

    try:
        driver.get(base_url)
        for cookie in raw_cookies:
            if not isinstance(cookie, dict):
                continue
            item = dict(cookie)
            item.pop("sameSite", None)
            try:
                driver.add_cookie(item)
            except Exception:
                continue
        driver.get(base_url)
        return True
    except Exception:
        return False


def _save_snow_session_cookies(driver: Any, *, cookie_path: Path) -> None:
    try:
        cookies = driver.get_cookies()
        cookie_path.parent.mkdir(parents=True, exist_ok=True)
        cookie_path.write_bytes(pickle.dumps(cookies))
    except Exception:
        return


def _attempt_snow_login(driver: Any, config: dict[str, Any]) -> None:
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys

    username = _safe_text(config.get("snow_username"))
    password = _safe_text(config.get("snow_password"))
    if not username or not password:
        return

    username_field = None
    for selector in (
        "input#user_name",
        "input[name='user_name']",
        "input[name='username']",
        "input[type='email']",
    ):
        elements = driver.find_elements(By.CSS_SELECTOR, selector)
        if elements:
            username_field = elements[0]
            break

    if username_field is None:
        return

    password_field = None
    for selector in (
        "input#user_password",
        "input[name='user_password']",
        "input[type='password']",
    ):
        elements = driver.find_elements(By.CSS_SELECTOR, selector)
        if elements:
            password_field = elements[0]
            break

    if password_field is None:
        return

    username_field.clear()
    username_field.send_keys(username)
    password_field.clear()
    password_field.send_keys(password)
    password_field.send_keys(Keys.ENTER)


def _find_elements_in_possible_contexts(driver: Any, css_selector: str) -> list[Any]:
    from selenium.webdriver.common.by import By

    try:
        driver.switch_to.default_content()
    except Exception:
        pass

    try:
        matches = driver.find_elements(By.CSS_SELECTOR, css_selector)
        if matches:
            return matches

        frames = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
        for frame in frames:
            try:
                driver.switch_to.default_content()
                driver.switch_to.frame(frame)
                matches = driver.find_elements(By.CSS_SELECTOR, css_selector)
                if matches:
                    return matches
            except Exception:
                continue
    except Exception:
        return []
    finally:
        try:
            driver.switch_to.default_content()
        except Exception:
            pass

    return []


def _is_incident_list_ready(driver: Any) -> bool:
    selectors = (
        "table.data_list_table",
        "table.list_table",
        "div.list2_body",
        "tr.list_row",
        "[role='grid']",
        "[role='table']",
        "a[href*='incident.do']",
    )
    for selector in selectors:
        if _find_elements_in_possible_contexts(driver, selector):
            return True
    return False


def _is_login_gate(driver: Any) -> bool:
    current_url = _safe_text(getattr(driver, "current_url", "")).lower()
    url_tokens = ("login", "saml", "oauth", "okta", "azure", "auth")
    if any(token in current_url for token in url_tokens) and "incident_list.do" not in current_url:
        return True

    login_selectors = (
        "input[type='password']",
        "input#user_password",
        "input[name='user_password']",
        "form[action*='login']",
    )
    for selector in login_selectors:
        if _find_elements_in_possible_contexts(driver, selector):
            return True

    return False


def _wait_for_interactive_sso(
    driver: Any,
    *,
    list_url: str,
    timeout_seconds: int,
    poll_seconds: int,
) -> bool:
    deadline = time.time() + max(timeout_seconds, 10)
    poll = max(poll_seconds, 1)

    print("ServiceNow SSO: complete login in the opened Chrome window. Waiting for incident list...")

    while time.time() < deadline:
        try:
            if _is_incident_list_ready(driver) and not _is_login_gate(driver):
                return True

            # Don't navigate away while user is mid-SSO login
            if not _is_login_gate(driver):
                driver.get(list_url)
                try:
                    _wait_for_document_ready(driver, min(15, max(timeout_seconds, 5)))
                except Exception:
                    pass
                if _is_incident_list_ready(driver) and not _is_login_gate(driver):
                    return True
        except Exception:
            pass

        time.sleep(poll)

    return False


def _extract_header_index_map(driver: Any) -> dict[str, int]:
    from selenium.webdriver.common.by import By

    headers = driver.find_elements(By.CSS_SELECTOR, "table.data_list_table thead th, table.list_table thead th, [role='columnheader']")
    index_map: dict[str, int] = {}
    for idx, header in enumerate(headers):
        text = normalize_match_text(header.text)
        if not text:
            continue
        if "number" in text and "number" not in index_map:
            index_map["number"] = idx
        if "short" in text and "description" in text and "short_description" not in index_map:
            index_map["short_description"] = idx
        if "opened" in text and "opened" not in index_map:
            index_map["opened"] = idx
        if "created" in text and "opened" not in index_map:
            index_map["opened"] = idx
        if "state" in text and "state" not in index_map:
            index_map["state"] = idx
        if "assigned" in text and "assigned_to" not in index_map:
            index_map["assigned_to"] = idx
        if "updated" in text and "updated" not in index_map:
            index_map["updated"] = idx
    return index_map


def _extract_incident_rows_in_current_context(driver: Any, *, max_incidents: int) -> tuple[list[dict[str, Any]], bool]:
    from selenium.webdriver.common.by import By

    def _build_incident_from_values(
        *,
        cell_values: list[str],
        row_text: str,
        incident_url: str,
        link_text: str,
        index_map: dict[str, int],
    ) -> dict[str, Any] | None:
        number = ""
        if "number" in index_map and index_map["number"] < len(cell_values):
            number = cell_values[index_map["number"]]
        if not number:
            number = link_text

        match = re.search(r"\bINC\d{5,}\b", number or row_text, flags=re.IGNORECASE)
        if not match:
            return None
        number = match.group(0).upper()

        short_description = ""
        if "short_description" in index_map and index_map["short_description"] < len(cell_values):
            short_description = cell_values[index_map["short_description"]]
        if not short_description:
            adaptive_lines = [line for line in row_text.split("|") if re.search(r"adaptive\s*shield", line, flags=re.IGNORECASE)]
            if adaptive_lines:
                short_description = adaptive_lines[0].strip()

        if not re.search(r"adaptive\s*shield", short_description or row_text, flags=re.IGNORECASE):
            return None

        opened_raw = cell_values[index_map["opened"]] if "opened" in index_map and index_map["opened"] < len(cell_values) else ""
        state_raw = cell_values[index_map["state"]] if "state" in index_map and index_map["state"] < len(cell_values) else ""
        assigned_to_raw = cell_values[index_map["assigned_to"]] if "assigned_to" in index_map and index_map["assigned_to"] < len(cell_values) else ""
        updated_raw = cell_values[index_map["updated"]] if "updated" in index_map and index_map["updated"] < len(cell_values) else ""

        return {
            "ticket_number": number,
            "short_description": short_description or row_text,
            "ticket_opened_datetime": opened_raw,
            "ticket_state": state_raw,
            "ticket_assigned_to": assigned_to_raw,
            "ticket_last_update_datetime": updated_raw,
            "ticket_last_note_source": "",
            "ticket_last_update_content": "",
            "ticket_is_closed": _is_closed_state(state_raw),
            "incident_url": incident_url,
            "raw_opened_value": opened_raw,
            "raw_updated_value": updated_raw,
        }

    incidents: list[dict[str, Any]] = []
    seen_numbers: set[str] = set()
    table_els = driver.find_elements(By.CSS_SELECTOR, "table.data_list_table, table.list_table, div.list2_body, tr.list_row, [role='grid'], [role='table'], a[href*='incident.do']")
    has_table = bool(table_els)
    logging.info("[SNOW] _extract_incident_rows_in_current_context — url=%s has_table=%s",
                 driver.current_url, has_table)

    # Log all iframes present to help diagnose iframe-wrapped lists
    iframes = driver.find_elements(By.CSS_SELECTOR, "iframe")
    if iframes:
        iframe_ids = [f.get_attribute("id") or f.get_attribute("name") or "?" for f in iframes]
        logging.info("[SNOW]   iframes found: %s", iframe_ids)

    index_map = _extract_header_index_map(driver)
    logging.info("[SNOW]   header_index_map=%s", index_map)

    rows = driver.find_elements(By.CSS_SELECTOR, "table.data_list_table tbody tr, table.list_table tbody tr, tr.list_row, [role='row']")
    logging.info("[SNOW]   raw row elements found: %d", len(rows))
    for row in rows[:max_incidents]:
        cells = row.find_elements(By.CSS_SELECTOR, "td, [role='gridcell']")
        if not cells:
            continue

        cell_values = [_safe_text(cell.text) for cell in cells]
        row_text = " | ".join([value for value in cell_values if value])

        link_elements = row.find_elements(By.CSS_SELECTOR, "a[href*='incident.do']")
        incident_url = _safe_text(link_elements[0].get_attribute("href")) if link_elements else ""
        link_text = _safe_text(link_elements[0].text) if link_elements else ""

        incident = _build_incident_from_values(
            cell_values=cell_values,
            row_text=row_text,
            incident_url=incident_url,
            link_text=link_text,
            index_map=index_map,
        )
        if not incident:
            continue
        if incident["ticket_number"] in seen_numbers:
            continue
        seen_numbers.add(incident["ticket_number"])
        incidents.append(incident)

    if incidents:
        return incidents, has_table

    # Fallback for modern ServiceNow lists rendered in shadow DOM.
    try:
        js_rows = driver.execute_script(
            """
const limit = Math.max(0, Number(arguments[0]) || 0);
const normalize = (value) => (value || '').replace(/\\s+/g, ' ').trim();
const results = [];
const seen = new Set();

function collectLinks(root) {
  if (!root || !root.querySelectorAll) {
    return [];
  }
  const links = Array.from(root.querySelectorAll("a[href*='incident.do']"));
  for (const element of root.querySelectorAll('*')) {
    if (element.shadowRoot) {
      links.push(...collectLinks(element.shadowRoot));
    }
  }
  return links;
}

for (const link of collectLinks(document)) {
  if (results.length >= limit) {
    break;
  }
  const href = normalize(link.href || link.getAttribute('href') || '');
  if (!href || seen.has(href)) {
    continue;
  }
  seen.add(href);

  const row = link.closest("tr, [role='row'], .list_row");
  let rowText = '';
  let cells = [];
  if (row) {
    rowText = normalize(row.innerText || row.textContent || '');
    cells = Array.from(row.querySelectorAll("td, [role='gridcell']"))
      .map((cell) => normalize(cell.innerText || cell.textContent || ''))
      .filter(Boolean);
  }

  results.push({
    href,
    link_text: normalize(link.innerText || link.textContent || ''),
    row_text: rowText,
    cells,
  });
}

return results;
            """,
            int(max_incidents),
        )
    except Exception as exc:
        logging.info("[SNOW]   shadow/js fallback failed: %s", exc)
        js_rows = []

    js_candidates = js_rows if isinstance(js_rows, list) else []
    logging.info("[SNOW]   shadow/js fallback candidates: %d", len(js_candidates))

    for payload in js_candidates[:max_incidents]:
        if not isinstance(payload, dict):
            continue
        cell_values = [_safe_text(value) for value in payload.get("cells", []) if _safe_text(value)]
        row_text = _safe_text(payload.get("row_text"))
        incident_url = _safe_text(payload.get("href"))
        link_text = _safe_text(payload.get("link_text"))
        incident = _build_incident_from_values(
            cell_values=cell_values,
            row_text=row_text,
            incident_url=incident_url,
            link_text=link_text,
            index_map=index_map,
        )
        if not incident:
            continue
        if incident["ticket_number"] in seen_numbers:
            continue
        seen_numbers.add(incident["ticket_number"])
        incidents.append(incident)

    if incidents:
        has_table = True

    return incidents, has_table


def _extract_incident_rows_from_current_page(driver: Any, *, max_incidents: int) -> list[dict[str, Any]]:
    try:
        driver.switch_to.default_content()
    except Exception:
        pass

    try:
        incidents, has_table = _extract_incident_rows_in_current_context(driver, max_incidents=max_incidents)
        if has_table and incidents:
            return incidents

        from selenium.webdriver.common.by import By

        frames = driver.find_elements(By.CSS_SELECTOR, "iframe, frame")
        for frame in frames:
            try:
                driver.switch_to.default_content()
                driver.switch_to.frame(frame)
                incidents, has_table = _extract_incident_rows_in_current_context(driver, max_incidents=max_incidents)
                if has_table and incidents:
                    return incidents
            except Exception:
                continue
    finally:
        try:
            driver.switch_to.default_content()
        except Exception:
            pass

    return []


def _snow_field_text(raw: Any, *, prefer_value: bool = False) -> str:
    if isinstance(raw, dict):
        first = "value" if prefer_value else "display_value"
        second = "display_value" if prefer_value else "value"
        return _safe_text(raw.get(first)) or _safe_text(raw.get(second))
    return _safe_text(raw)


def _build_snow_lookback_query(
    base_query: str,
    run_ts_utc: datetime,
    lookback_days: int,
    *,
    field: str = "opened_at",
) -> str:
    filter_field = _safe_text(field) or "opened_at"
    start_dt = run_ts_utc - timedelta(days=max(0, _safe_int(lookback_days, default=0)))
    start_value = start_dt.strftime("%Y-%m-%d %H:%M:%S")
    end_value = run_ts_utc.strftime("%Y-%m-%d %H:%M:%S")
    window_clause = f"{filter_field}>={start_value}^{filter_field}<={end_value}"
    stripped = _safe_text(base_query)
    if not stripped:
        return window_clause
    return f"{stripped}^{window_clause}"


def _collect_snow_incidents_via_api(
    *,
    driver: Any,
    base_url: str,
    query: str,
    max_incidents: int,
    timeout_seconds: int,
    cache_store: WebCacheStore | None = None,
    cache_flags: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    endpoint = f"{base_url}/api/now/table/incident"
    session = requests.Session()

    user_agent = ""
    try:
        user_agent = _safe_text(driver.execute_script("return navigator.userAgent || '';"))
    except Exception:
        user_agent = ""
    if user_agent:
        session.headers.update({"User-Agent": user_agent})

    for cookie in driver.get_cookies():
        if not isinstance(cookie, dict):
            continue
        name = _safe_text(cookie.get("name"))
        value = _safe_text(cookie.get("value"))
        domain = _safe_text(cookie.get("domain")).lstrip(".")
        path = _safe_text(cookie.get("path")) or "/"
        if not name or not value:
            continue
        try:
            if domain:
                session.cookies.set(name, value, domain=domain, path=path)
            else:
                session.cookies.set(name, value, path=path)
        except Exception:
            continue

    rows: list[dict[str, Any]] = []
    offset = 0
    page_size = min(200, max(1, min(max_incidents, 200)))

    while len(rows) < max_incidents:
        limit = min(page_size, max_incidents - len(rows))
        params = {
            "sysparm_query": query,
            "sysparm_limit": str(limit),
            "sysparm_offset": str(offset),
            "sysparm_fields": "number,short_description,opened_at,state,assigned_to,sys_updated_on,sys_id",
            "sysparm_display_value": "all",
            "sysparm_exclude_reference_link": "true",
        }
        page_cache_key = _cache_key("snow_api_page", endpoint, params)
        page_payload: dict[str, Any] | None = None

        if cache_store and _is_cache_enabled(cache_flags, "snow_list"):
            cached_page = cache_store.get("snow_list", page_cache_key)
            if isinstance(cached_page, dict):
                page_payload = cached_page
                logging.info("[CACHE][SNOW] hit mode=api offset=%s", params["sysparm_offset"])

        if page_payload is None:
            response = session.get(endpoint, params=params, timeout=max(10, timeout_seconds))
            if response.status_code in {401, 403}:
                logging.info("[SNOW][API] unauthorized (%s), fallback to DOM", response.status_code)
                return []
            response.raise_for_status()
            api_payload = response.json() if response.content else {}
            page_payload = {
                "mode": "api",
                "endpoint": endpoint,
                "params": params,
                "api_payload": api_payload,
            }
            if cache_store and _is_cache_enabled(cache_flags, "snow_list"):
                cache_store.set("snow_list", page_cache_key, page_payload, metadata={"mode": "api"})

        api_payload = page_payload.get("api_payload", {}) if isinstance(page_payload, dict) else {}
        records = api_payload.get("result", []) if isinstance(api_payload, dict) else []
        if not isinstance(records, list):
            logging.info("[SNOW][API] unexpected payload, fallback to DOM")
            return []

        logging.info("[SNOW][API] fetched=%d offset=%d", len(records), offset)
        if not records:
            break

        for record in records:
            if not isinstance(record, dict):
                continue

            number = _snow_field_text(record.get("number"))
            short_description = _snow_field_text(record.get("short_description"))
            if not number or not re.search(r"\bINC\d{5,}\b", number, flags=re.IGNORECASE):
                continue
            if not re.search(r"adaptive\s*shield", short_description, flags=re.IGNORECASE):
                continue

            sys_id = _snow_field_text(record.get("sys_id"), prefer_value=True)
            opened_raw = _snow_field_text(record.get("opened_at"), prefer_value=True)
            state_raw = _snow_field_text(record.get("state"))
            assigned_to_raw = _snow_field_text(record.get("assigned_to"))
            updated_raw = _snow_field_text(record.get("sys_updated_on"), prefer_value=True)
            incident_url = f"{base_url}/incident.do?sys_id={sys_id}" if sys_id else ""

            rows.append(
                {
                    "ticket_number": number.upper(),
                    "short_description": short_description,
                    "ticket_opened_datetime": opened_raw,
                    "ticket_state": state_raw,
                    "ticket_assigned_to": assigned_to_raw,
                    "ticket_last_update_datetime": updated_raw,
                    "ticket_last_note_source": "",
                    "ticket_last_update_content": "",
                    "ticket_is_closed": _is_closed_state(state_raw),
                    "incident_url": incident_url,
                    "raw_opened_value": opened_raw,
                    "raw_updated_value": updated_raw,
                }
            )
            if len(rows) >= max_incidents:
                break

        if len(records) < limit:
            break
        offset += len(records)

    return rows


def _collect_snow_incidents(
    *,
    config: dict[str, Any],
    lookback_days: int,
    run_ts_utc: datetime,
    pipeline_errors: list[dict[str, Any]] | None,
) -> list[dict[str, Any]]:
    base_url = _safe_text(config.get("snow_base_url")).rstrip("/")
    if not base_url:
        _append_pipeline_error(
            pipeline_errors,
            stage="servicenow",
            message="SNOW_BASE_URL is required when SNOW_ENABLED=true",
        )
        return []

    timeout_seconds = _safe_int(config.get("snow_login_timeout_seconds"), default=20)
    max_incidents = _safe_int(config.get("snow_max_incidents"), default=200)
    fetch_detail_notes = bool(config.get("snow_fetch_detail_notes", True))
    interactive_login = bool(config.get("snow_interactive_login", True))
    manual_timeout_seconds = _safe_int(config.get("snow_manual_login_timeout_seconds"), default=300)
    manual_poll_seconds = _safe_int(config.get("snow_manual_login_poll_seconds"), default=3)
    open_baseurl_first = bool(config.get("snow_open_baseurl_first", True))
    use_table_api = bool(config.get("snow_use_table_api", True))
    table_api_only = bool(config.get("snow_table_api_only", False))
    date_filter_enabled = bool(config.get("snow_date_filter_enabled", True))
    configured_filter_field = _safe_text(config.get("snow_date_filter_field")) or "opened_at"
    date_filter_field = "opened_at"
    if configured_filter_field and configured_filter_field != "opened_at":
        logging.info(
            "[SNOW] SNOW_DATE_FILTER_FIELD=%r is not supported in this version; fallback to opened_at",
            configured_filter_field,
        )
    if not date_filter_enabled:
        logging.info(
            "[SNOW] SNOW_DATE_FILTER_ENABLED=false ignored; URL query still applies %s-day lookback on %s",
            lookback_days,
            date_filter_field,
        )

    cache_store = config.get("cache_store") if isinstance(config.get("cache_store"), WebCacheStore) else None
    cache_flags = config.get("cache_flags") if isinstance(config.get("cache_flags"), dict) else {}

    try:
        driver = _build_snow_driver(config)
    except Exception as exc:
        _append_pipeline_error(
            pipeline_errors,
            stage="servicenow",
            message=f"Failed to initialize Selenium driver: {exc}",
        )
        return []

    incidents: list[dict[str, Any]] = []
    cookie_path = Path(_safe_text(config.get("snow_cookie_path")) or str(Path.cwd() / ".snow_cookies.pkl"))
    base_query = _safe_text(config.get("snow_incident_query")) or "short_descriptionLIKEAdaptiveShield"
    query = _build_snow_lookback_query(base_query, run_ts_utc, lookback_days, field=date_filter_field)
    encoded_query = quote(query, safe="^=><")
    list_url = f"{base_url}/incident_list.do?sysparm_query={encoded_query}"
    logging.info("[SNOW] Incident list query (with lookback): %s", query)
    cutoff_dt = run_ts_utc - timedelta(days=lookback_days)

    try:
        handles_before = set(driver.window_handles)
        if open_baseurl_first:
            try:
                driver.get(base_url)
                _wait_for_document_ready(driver, timeout_seconds)
            except Exception:
                pass
        driver.get(list_url)
        try:
            _wait_for_document_ready(driver, timeout_seconds)
        except Exception:
            pass

        new_handles = set(driver.window_handles) - handles_before
        if new_handles:
            driver.switch_to.window(list(new_handles)[-1])
            logging.info("[SNOW] Switched to new tab opened by ServiceNow: %s", driver.current_url)
            try:
                _wait_for_document_ready(driver, timeout_seconds)
            except Exception:
                pass

        logging.info("[SNOW] After navigation — url=%s title=%r handles=%d", driver.current_url, driver.title, len(driver.window_handles))

        if _is_login_gate(driver):
            _attempt_snow_login(driver, config)
            try:
                _wait_for_document_ready(driver, timeout_seconds)
            except Exception:
                pass

        if _is_login_gate(driver):
            if interactive_login:
                ok = _wait_for_interactive_sso(
                    driver,
                    list_url=list_url,
                    timeout_seconds=manual_timeout_seconds,
                    poll_seconds=manual_poll_seconds,
                )
                if not ok:
                    _append_pipeline_error(
                        pipeline_errors,
                        stage="servicenow",
                        message="ServiceNow interactive SSO login timed out before incident list became available",
                    )
                    return []
            else:
                _append_pipeline_error(
                    pipeline_errors,
                    stage="servicenow",
                    message="ServiceNow login required. Enable SNOW_INTERACTIVE_LOGIN or provide SNOW_USERNAME/SNOW_PASSWORD",
                )
                return []

        rows: list[dict[str, Any]] = []
        if use_table_api:
            try:
                rows = _collect_snow_incidents_via_api(
                    driver=driver,
                    base_url=base_url,
                    query=query,
                    max_incidents=max_incidents,
                    timeout_seconds=timeout_seconds,
                    cache_store=cache_store,
                    cache_flags=cache_flags,
                )
                logging.info("[SNOW][API] extracted rows: %d", len(rows))
            except Exception as exc:
                logging.info("[SNOW][API] failed, fallback to DOM: %s", exc)
                rows = []

        if table_api_only and not rows:
            _append_pipeline_error(
                pipeline_errors,
                stage="servicenow",
                message="ServiceNow Table API returned no incidents while SNOW_TABLE_API_ONLY=true",
            )
            return []

        if not rows:
            table_selector = "table.data_list_table, table.list_table, tr.list_row, [role='grid'], [role='table'], a[href*='incident.do'], iframe, frame"
            try:
                from selenium.webdriver.support.ui import WebDriverWait
                from selenium.webdriver.support import expected_conditions as EC
                from selenium.webdriver.common.by import By as _By

                WebDriverWait(driver, timeout_seconds).until(
                    EC.presence_of_element_located((_By.CSS_SELECTOR, table_selector))
                )
                logging.info("[SNOW] Table/iframe detected in DOM")
            except Exception:
                logging.warning("[SNOW] Timed out waiting for table — proceeding anyway")
                logging.warning("[SNOW]   url=%s", driver.current_url)
                logging.warning("[SNOW]   title=%r", driver.title)
                try:
                    logging.warning("[SNOW]   page_source_preview=%s", driver.page_source[:2000])
                except Exception:
                    pass

            page_first_row = 1
            page_size = 20
            while len(rows) < max_incidents:
                logging.info("[SNOW] Page starting at row %d — url=%s", page_first_row, driver.current_url)
                page_cache_key = _cache_key("snow_dom_page", driver.current_url, page_first_row, max_incidents - len(rows))
                page_rows: list[dict[str, Any]] = []

                if cache_store and _is_cache_enabled(cache_flags, "snow_list"):
                    cached_page = cache_store.get("snow_list", page_cache_key)
                    if isinstance(cached_page, dict) and isinstance(cached_page.get("rows"), list):
                        page_rows = [item for item in cached_page["rows"] if isinstance(item, dict)]
                        logging.info("[CACHE][SNOW] hit mode=dom row=%d", page_first_row)

                if not page_rows:
                    page_rows = _extract_incident_rows_from_current_page(driver, max_incidents=max_incidents - len(rows))

                html_path = ""
                if cache_store and _is_cache_enabled(cache_flags, "snow_html"):
                    try:
                        html_path = cache_store.set_html(
                            "snow_html",
                            _cache_key("snow_list_html", driver.current_url, page_first_row),
                            driver.page_source,
                            metadata={"stage": "list", "page_first_row": page_first_row, "url": driver.current_url},
                        )
                    except Exception:
                        html_path = ""

                if cache_store and _is_cache_enabled(cache_flags, "snow_list"):
                    cache_store.set(
                        "snow_list",
                        page_cache_key,
                        {
                            "mode": "dom",
                            "url": driver.current_url,
                            "query": query,
                            "page_first_row": page_first_row,
                            "rows": page_rows,
                            "html_gz_path": html_path,
                        },
                        metadata={"mode": "dom"},
                    )

                logging.info("[SNOW]   page rows extracted: %d  cumulative: %d", len(page_rows), len(rows) + len(page_rows))
                if page_rows:
                    page_size = len(page_rows)
                rows.extend(page_rows)

                if len(page_rows) < page_size or len(rows) >= max_incidents:
                    logging.info("[SNOW] Pagination done — total rows: %d", len(rows))
                    break

                page_first_row += page_size
                from urllib.parse import urlparse, urlencode, parse_qs, urlunparse

                parsed = urlparse(list_url)
                qs = parse_qs(parsed.query, keep_blank_values=True)
                qs["sysparm_first_row"] = [str(page_first_row)]
                next_url = urlunparse(parsed._replace(query=urlencode(qs, doseq=True)))
                driver.get(next_url)
                try:
                    _wait_for_document_ready(driver, timeout_seconds)
                except Exception:
                    pass

        print(f"ServiceNow: extracted {len(rows)} incident rows")

        for row in rows:
            opened_dt = _parse_datetime_utc(row.get("ticket_opened_datetime"))
            if opened_dt is None:
                _append_pipeline_error(
                    pipeline_errors,
                    stage="servicenow",
                    message=f"Skip incident due to unparseable opened datetime: {row.get('ticket_number')}",
                    ticket_number=_safe_text(row.get("ticket_number")),
                )
                continue
            if opened_dt < cutoff_dt:
                continue

            row["ticket_opened_datetime"] = opened_dt.strftime("%Y-%m-%dT%H:%M:%SZ")

            if fetch_detail_notes:
                note_text, note_source, detail_updated = _extract_incident_last_note(
                    driver,
                    _safe_text(row.get("incident_url")),
                    timeout_seconds,
                    cache_store=cache_store,
                    cache_flags=cache_flags,
                )
                if note_text:
                    row["ticket_last_update_content"] = note_text
                    row["ticket_last_note_source"] = note_source
                if detail_updated:
                    row["ticket_last_update_datetime"] = detail_updated

            last_update_dt = _parse_datetime_utc(row.get("ticket_last_update_datetime")) or opened_dt
            row["ticket_last_update_datetime"] = last_update_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
            row["ticket_last_update_days_ago"] = compute_days_ago(run_ts_utc, last_update_dt)
            row["ticket_is_closed"] = _is_closed_state(row.get("ticket_state"))
            incidents.append(row)

        _save_snow_session_cookies(driver, cookie_path=cookie_path)

    except Exception as exc:
        _append_pipeline_error(
            pipeline_errors,
            stage="servicenow",
            message=f"ServiceNow scraping failed: {exc}",
        )
    finally:
        if not config.get("snow_keep_browser_open", False):
            try:
                driver.quit()
            except Exception:
                pass

    incidents.sort(
        key=lambda row: _parse_datetime_utc(row.get("ticket_last_update_datetime"))
        or _parse_datetime_utc(row.get("ticket_opened_datetime"))
        or datetime(1970, 1, 1, tzinfo=timezone.utc),
        reverse=True,
    )
    return incidents


def _empty_snow_outputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    row_df = pd.DataFrame(columns=[JOIN_KEY, *SNOW_COLUMNS])
    raw_df = pd.DataFrame(columns=SNOW_RAW_COLUMNS)
    details_df = pd.DataFrame(columns=SNOW_DETAIL_COLUMNS)
    return row_df, raw_df, details_df


def _map_incidents_to_summary(
    summary_df: pd.DataFrame,
    incidents: list[dict[str, Any]],
    *,
    run_ts_utc: datetime,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if summary_df.empty:
        return _empty_snow_outputs()

    summary_records = summary_df.to_dict("records")
    key_candidates: list[str] = []
    alert_ids_by_key: dict[str, list[str]] = defaultdict(list)

    for row in summary_records:
        key = build_check_key(
            row.get("saas_name") or row.get("integration_name"),
            row.get("integration_alias") or row.get("integration_name"),
            row.get("security_check_name") or row.get("security_check_id"),
        )
        if not key:
            continue
        if key not in key_candidates:
            key_candidates.append(key)
        alert_id = _safe_text(row.get(JOIN_KEY))
        if alert_id:
            alert_ids_by_key[key].append(alert_id)

    incidents_by_key: dict[str, list[dict[str, Any]]] = defaultdict(list)
    raw_rows: list[dict[str, Any]] = []

    for incident in incidents:
        incident_row = dict(incident)
        parsed_key = parse_short_description_key(incident_row.get("short_description"))
        match_mode = ""

        if parsed_key and parsed_key in alert_ids_by_key:
            ticket_key = parsed_key
            match_mode = "structured"
        else:
            ticket_key = _fallback_match_key(incident_row.get("short_description"), key_candidates)
            match_mode = "fallback" if ticket_key else "unmatched"

        incident_row["ticket_match_key"] = ticket_key
        incident_row["match_mode"] = match_mode

        if ticket_key:
            incidents_by_key[ticket_key].append(incident_row)

        raw_rows.append(
            {
                "ticket_number": _safe_text(incident_row.get("ticket_number")),
                "short_description": _safe_text(incident_row.get("short_description")),
                "ticket_opened_datetime": _to_iso_utc(incident_row.get("ticket_opened_datetime")),
                "ticket_state": _safe_text(incident_row.get("ticket_state")),
                "ticket_assigned_to": _safe_text(incident_row.get("ticket_assigned_to")),
                "ticket_last_update_datetime": _to_iso_utc(incident_row.get("ticket_last_update_datetime")),
                "ticket_last_update_days_ago": compute_days_ago(run_ts_utc, incident_row.get("ticket_last_update_datetime")),
                "ticket_last_note_source": _safe_text(incident_row.get("ticket_last_note_source")),
                "ticket_last_update_content": _safe_text(incident_row.get("ticket_last_update_content")),
                "ticket_is_closed": bool(incident_row.get("ticket_is_closed")),
                "ticket_match_key": ticket_key,
                "match_mode": match_mode,
                "incident_url": _safe_text(incident_row.get("incident_url")),
                "raw_opened_value": _safe_text(incident_row.get("raw_opened_value")),
                "raw_updated_value": _safe_text(incident_row.get("raw_updated_value")),
            }
        )

    for key, rows in incidents_by_key.items():
        rows.sort(
            key=lambda row: _parse_datetime_utc(row.get("ticket_last_update_datetime"))
            or _parse_datetime_utc(row.get("ticket_opened_datetime"))
            or datetime(1970, 1, 1, tzinfo=timezone.utc),
            reverse=True,
        )

    details_rows: list[dict[str, Any]] = []
    for key, rows in incidents_by_key.items():
        total = len(rows)
        open_count = sum(1 for row in rows if not bool(row.get("ticket_is_closed")))
        for row in rows:
            details_rows.append(
                {
                    "ticket_match_key": key,
                    "ticket_number": _safe_text(row.get("ticket_number")),
                    "ticket_url": _safe_text(row.get("incident_url")),
                    "ticket_opened_datetime": _to_iso_utc(row.get("ticket_opened_datetime")),
                    "ticket_state": _safe_text(row.get("ticket_state")),
                    "ticket_assigned_to": _safe_text(row.get("ticket_assigned_to")),
                    "ticket_owner": _safe_text(row.get("ticket_assigned_to")),
                    "ticket_status": _safe_text(row.get("ticket_state")),
                    "ticket_last_update_datetime": _to_iso_utc(row.get("ticket_last_update_datetime")),
                    "ticket_last_update_days_ago": compute_days_ago(run_ts_utc, row.get("ticket_last_update_datetime")),
                    "ticket_last_note_source": _safe_text(row.get("ticket_last_note_source")),
                    "ticket_last_update_content": _safe_text(row.get("ticket_last_update_content")),
                    "ticket_is_closed": bool(row.get("ticket_is_closed")),
                    "ticket_count_for_check": total,
                    "open_ticket_count_for_check": open_count,
                }
            )

    row_rows: list[dict[str, Any]] = []
    for row in summary_records:
        alert_id = _safe_text(row.get(JOIN_KEY))
        key = build_check_key(
            row.get("saas_name") or row.get("integration_name"),
            row.get("integration_alias") or row.get("integration_name"),
            row.get("security_check_name") or row.get("security_check_id"),
        )
        matched_rows = incidents_by_key.get(key, [])

        if matched_rows:
            latest = matched_rows[0]
            ticket_count = len(matched_rows)
            open_count = sum(1 for item in matched_rows if not bool(item.get("ticket_is_closed")))

            row_rows.append(
                {
                    JOIN_KEY: alert_id,
                    "ticket_number": _safe_text(latest.get("ticket_number")),
                    "ticket_opened_datetime": _to_iso_utc(latest.get("ticket_opened_datetime")),
                    "ticket_state": _safe_text(latest.get("ticket_state")),
                    "ticket_assigned_to": _safe_text(latest.get("ticket_assigned_to")),
                    "ticket_owner": _safe_text(latest.get("ticket_assigned_to")),
                    "ticket_status": _safe_text(latest.get("ticket_state")),
                    "ticket_last_update_datetime": _to_iso_utc(latest.get("ticket_last_update_datetime")),
                    "ticket_last_update_days_ago": compute_days_ago(run_ts_utc, latest.get("ticket_last_update_datetime")),
                    "ticket_last_note_source": _safe_text(latest.get("ticket_last_note_source")),
                    "ticket_last_update_content": _safe_text(latest.get("ticket_last_update_content")),
                    "ticket_is_closed": bool(latest.get("ticket_is_closed")),
                    "ticket_match_key": key,
                    "ticket_count_for_check": ticket_count,
                    "open_ticket_count_for_check": open_count,
                }
            )
        else:
            row_rows.append({JOIN_KEY: alert_id, **{column: None for column in SNOW_COLUMNS}})

    row_df = pd.DataFrame(row_rows, columns=[JOIN_KEY, *SNOW_COLUMNS])
    raw_df = pd.DataFrame(raw_rows, columns=SNOW_RAW_COLUMNS)
    details_df = pd.DataFrame(details_rows, columns=SNOW_DETAIL_COLUMNS)

    return row_df, raw_df, details_df


def fetch_related_tickets(
    summary_df: pd.DataFrame,
    lookback_days: int,
    *,
    config: dict[str, Any] | None = None,
    run_ts_utc: datetime | None = None,
    pipeline_errors: list[dict[str, Any]] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if summary_df is None or summary_df.empty:
        return _empty_snow_outputs()

    runtime_config = config or {}
    if not bool(runtime_config.get("snow_enabled", False)):
        return _empty_snow_outputs()

    effective_run_ts = run_ts_utc or datetime.now(timezone.utc)
    incidents = _collect_snow_incidents(
        config=runtime_config,
        lookback_days=lookback_days,
        run_ts_utc=effective_run_ts,
        pipeline_errors=pipeline_errors,
    )

    return _map_incidents_to_summary(summary_df, incidents, run_ts_utc=effective_run_ts)


def merge_snow_columns(summary_df: pd.DataFrame, snow_df: pd.DataFrame | None) -> pd.DataFrame:
    result = summary_df.copy()
    for column in SNOW_COLUMNS:
        if column not in result.columns:
            result[column] = None

    if snow_df is None or snow_df.empty:
        return result
    if JOIN_KEY not in snow_df.columns:
        return result

    selected_columns = [JOIN_KEY] + [col for col in SNOW_COLUMNS if col in snow_df.columns]
    merged = result.drop(columns=SNOW_COLUMNS, errors="ignore").merge(
        snow_df[selected_columns].drop_duplicates(subset=[JOIN_KEY], keep="last"),
        on=JOIN_KEY,
        how="left",
    )

    for column in SNOW_COLUMNS:
        if column not in merged.columns:
            merged[column] = None

    return merged


def export_all(
    summary_df: pd.DataFrame | None,
    entities_df: pd.DataFrame | None,
    errors_df: pd.DataFrame | None,
    output_dir: str,
    ts: str,
    *,
    snow_ticket_details_df: pd.DataFrame | None = None,
    export_xlsx: bool = True,
    export_csv: bool = True,
) -> dict[str, str]:
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    summary_df = summary_df if summary_df is not None else pd.DataFrame()
    entities_df = entities_df if entities_df is not None else pd.DataFrame()
    errors_df = errors_df if errors_df is not None else pd.DataFrame()
    snow_ticket_details_df = (
        snow_ticket_details_df if snow_ticket_details_df is not None else pd.DataFrame(columns=SNOW_DETAIL_COLUMNS)
    )

    results: dict[str, Any] = {
        "xlsx_path": "",
        "summary_csv_path": "",
        "entities_csv_path": "",
        "errors_csv_path": "",
        "snow_tickets_csv_path": "",
    }

    if export_xlsx:
        xlsx_path = output_path / f"AS_Weekly_Report_{ts}.xlsx"
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            summary_df.to_excel(writer, sheet_name="summary", index=False)
            entities_df.to_excel(writer, sheet_name="entities", index=False)
            errors_df.to_excel(writer, sheet_name="errors", index=False)
            snow_ticket_details_df.to_excel(writer, sheet_name="servicenow_tickets", index=False)
        results["xlsx_path"] = str(xlsx_path)

    if export_csv:
        summary_csv_path = output_path / f"AS_Weekly_Summary_{ts}.csv"
        entities_csv_path = output_path / f"AS_Weekly_Entities_{ts}.csv"
        errors_csv_path = output_path / f"AS_Weekly_Errors_{ts}.csv"
        snow_tickets_csv_path = output_path / f"AS_Weekly_ServiceNow_Tickets_{ts}.csv"

        summary_df.to_csv(summary_csv_path, index=False)
        entities_df.to_csv(entities_csv_path, index=False)
        errors_df.to_csv(errors_csv_path, index=False)
        snow_ticket_details_df.to_csv(snow_tickets_csv_path, index=False)

        results["summary_csv_path"] = str(summary_csv_path)
        results["entities_csv_path"] = str(entities_csv_path)
        results["errors_csv_path"] = str(errors_csv_path)
        results["snow_tickets_csv_path"] = str(snow_tickets_csv_path)

    return {k: str(v) for k, v in results.items()}



STATUS_ORDER = ["failed", "degraded", "drifted", "pending", "dismissed", "passed", "unknown"]

def normalize_status(raw: Any) -> str:
    if raw is None:
        return "unknown"
    value = str(raw).strip().lower()
    if not value:
        return "unknown"
    if "fail" in value:
        return "failed"
    if "pass" in value:
        return "passed"
    if "degrad" in value:
        return "degraded"
    if "drift" in value:
        return "drifted"
    if "dismiss" in value:
        return "dismissed"
    if "pend" in value:
        return "pending"
    return value


def _parse_ts(ts: Any) -> datetime:
    if ts is None:
        return datetime.min.replace(tzinfo=timezone.utc)
    raw = str(ts).strip()
    if not raw:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(raw.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


def _escape(value: Any) -> str:
    import html
    return html.escape(str(value if value is not None else ""))


def _safe_json_load(raw: Any) -> Any:
    if raw in (None, "", "null"):
        return None
    if isinstance(raw, (dict, list, int, float, bool)):
        return raw
    try:
        return json.loads(str(raw))
    except Exception:
        return raw


def _json_pretty_html(raw: Any) -> str:
    value = _safe_json_load(raw)
    if value in (None, "", [], {}):
        return "<span class=\"as-muted\">N/A</span>"
    try:
        text = json.dumps(value, ensure_ascii=False, indent=2, default=str)
    except Exception:
        text = str(value)
    return f'<pre class="as-json">{_escape(text)}</pre>'


def _is_populated(value: Any) -> bool:
    return value not in (None, "", [], {}, ())


def _collect_matching_values(
    obj: Any,
    *,
    key_pattern: re.Pattern[str],
    value_checker: Callable[[Any], bool] | None = None,
    prefix: str = "",
) -> list[tuple[str, Any]]:
    collected: list[tuple[str, Any]] = []

    def walk(value: Any, path: str) -> None:
        if isinstance(value, dict):
            for key, sub in value.items():
                next_path = f"{path}.{key}" if path else str(key)
                walk(sub, next_path)
            return
        if isinstance(value, list):
            for idx, sub in enumerate(value):
                next_path = f"{path}[{idx}]"
                walk(sub, next_path)
            return
        if not _is_populated(value):
            return
        last_key = path.split(".")[-1] if path else ""
        key_match = bool(key_pattern.search(last_key.lower()))
        value_match = bool(value_checker(value)) if value_checker else False
        if key_match or value_match:
            collected.append((path or prefix or "value", value))

    walk(obj, prefix)
    return collected


def _entity_preview_model(entity: dict[str, Any]) -> dict[str, Any]:
    name = str(entity.get("entity_name") or "").strip()

    raw_sources = [
        entity.get("entity_raw"),
        entity.get("entity_usage"),
        entity.get("entity_extra_context"),
    ]

    emails: list[str] = []
    if "@" in name:
        emails.append(name)

    email_pattern = re.compile(r"email", re.I)

    for source in raw_sources:
        for _, value in _collect_matching_values(
            source,
            key_pattern=email_pattern,
            value_checker=lambda v: isinstance(v, str) and "@" in v,
        ):
            as_text = str(value).strip()
            if as_text and as_text not in emails:
                emails.append(as_text)

    date_keys = re.compile(r"date|time|last|created|updated|seen|expiration|expiry", re.I)
    date_entries: list[str] = []

    direct_date = entity.get("entity_dismiss_expiration_date")
    if _is_populated(direct_date):
        date_entries.append(f"dismiss_expiration: {direct_date}")

    for source_name, source in (
        ("usage", entity.get("entity_usage")),
        ("extra_context", entity.get("entity_extra_context")),
        ("raw", entity.get("entity_raw")),
    ):
        for path, value in _collect_matching_values(source, key_pattern=date_keys):
            entry = f"{source_name}.{path}: {value}"
            if entry not in date_entries:
                date_entries.append(entry)

    return {
        "name": name,
        "emails": emails[:5],
        "dates": date_entries[:8],
    }


def _entity_other_payload(entity: dict[str, Any]) -> dict[str, Any]:
    payload: dict[str, Any] = {}
    candidate = {
        "entity_type": entity.get("entity_type"),
        "entity_dismissed": entity.get("entity_dismissed"),
        "entity_dismissed_reason": entity.get("entity_dismissed_reason"),
        "entity_dismiss_expiration_date": entity.get("entity_dismiss_expiration_date"),
        "entity_extra_context": entity.get("entity_extra_context"),
        "entity_usage": entity.get("entity_usage"),
        "entity_raw": entity.get("entity_raw"),
    }
    for key, value in candidate.items():
        if _is_populated(value):
            payload[key] = value
    return payload


def _style_badge(text: str, kind: str) -> str:
    return f'<span class="as-badge as-badge-{kind}">{_escape(text)}</span>'


def _safe_http_url(raw: Any) -> str:
    text = _safe_text(raw)
    if not text:
        return ""
    parsed = urlparse(text)
    if parsed.scheme in {"http", "https"} and parsed.netloc:
        return text
    return ""


def _html_link(label: str, href: Any) -> str:
    safe_href = _safe_http_url(href)
    if not safe_href:
        return ""
    return (
        f'<a href="{_escape(safe_href)}" target="_blank" rel="noopener noreferrer">'
        f'{_escape(label)}</a>'
    )


def _build_as_check_details_link(item: dict[str, Any]) -> str:
    security_check = _safe_text(item.get("security_check_id")) or _safe_text(item.get("security_check_name"))
    if not security_check:
        api_link = _safe_text(item.get("security_check_api_link"))
        if api_link:
            match = re.search(r"/security_checks/([^/?#]+)", api_link)
            if match:
                security_check = _safe_text(match.group(1))

    if not security_check:
        return ""

    exposure = quote(security_check, safe="")
    return (
        "https://dashboard.adaptive-shield.com/security_checks"
        f"?exposure={exposure}&exposureTab=overview"
    )


def _build_latest_snow_ticket_link(latest_ticket: dict[str, Any] | None) -> str:
    if not isinstance(latest_ticket, dict):
        return ""

    direct_url = _safe_http_url(latest_ticket.get("ticket_url"))
    if direct_url:
        return direct_url

    ticket_number = _safe_text(latest_ticket.get("ticket_number"))
    if not ticket_number:
        return ""

    runtime_config = globals().get("config")
    snow_base_url = ""
    if isinstance(runtime_config, dict):
        snow_base_url = _safe_text(runtime_config.get("snow_base_url"))
    if not snow_base_url:
        return ""
    snow_base_url = snow_base_url.rstrip("/")
    number_enc = quote(ticket_number, safe="")
    return f"{snow_base_url}/incident_list.do?sysparm_query=number={number_enc}"


def _format_count(value: Any, default: str = "0") -> str:
    if value in (None, ""):
        return default
    if isinstance(value, bool):
        return "1" if value else "0"
    if isinstance(value, (int, float)):
        numeric = float(value)
        if not math.isfinite(numeric):
            return default
        return str(int(numeric)) if numeric.is_integer() else str(numeric)

    text = str(value).strip()
    if not text:
        return default
    if text.lower() == "global":
        return "global"

    try:
        numeric = float(text)
    except ValueError:
        return text

    if not math.isfinite(numeric):
        return default
    return str(int(numeric)) if numeric.is_integer() else text


def _status_rank(status: str) -> int:
    try:
        return STATUS_ORDER.index(status)
    except ValueError:
        return len(STATUS_ORDER)


def _detect_flip_flop(statuses: list[str]) -> bool:
    clean = [normalize_status(s) for s in statuses if s is not None]
    if len(set(clean)) < 2:
        return False
    has_failed = any(s == "failed" for s in clean)
    has_passed = any(s == "passed" for s in clean)
    return has_failed and has_passed


def build_drift_groups(summary_df: pd.DataFrame, entities_df: pd.DataFrame) -> list[dict[str, Any]]:
    if summary_df.empty:
        return []

    groups: dict[tuple[str, str, str], list[dict[str, Any]]] = defaultdict(list)
    for row in summary_df.to_dict("records"):
        account_id = str(row.get("account_id") or "")
        integration_key = (
            str(row.get("integration_id") or "")
            or str(row.get("integration_alias") or "")
            or str(row.get("integration_name") or "")
            or "unknown_integration"
        )
        security_key = str(row.get("security_check_id") or row.get("security_check_name") or "unknown_check")
        groups[(account_id, integration_key, security_key)].append(row)

    entity_map: dict[str, list[dict[str, Any]]] = defaultdict(list)
    if not entities_df.empty:
        for entity in entities_df.to_dict("records"):
            alert_id = str(entity.get("alert_id") or "")
            if not alert_id:
                continue
            entity_map[alert_id].append(
                {
                    "entity_name": entity.get("entity_name"),
                    "entity_type": entity.get("entity_type"),
                    "entity_dismissed": entity.get("entity_dismissed"),
                    "entity_dismissed_reason": entity.get("entity_dismissed_reason"),
                    "entity_dismiss_expiration_date": entity.get("entity_dismiss_expiration_date"),
                    "entity_extra_context": _safe_json_load(entity.get("entity_extra_context_json")),
                    "entity_usage": _safe_json_load(entity.get("entity_usage_json")),
                    "entity_raw": _safe_json_load(entity.get("entity_raw_json")),
                }
            )

    result: list[dict[str, Any]] = []
    for (_, _, _), rows in groups.items():
        rows_sorted = sorted(rows, key=lambda r: _parse_ts(r.get("change_datetime")), reverse=True)
        current = rows_sorted[0]
        current_status = normalize_status(current.get("current_status"))
        statuses = [normalize_status(r.get("current_status")) for r in rows_sorted]
        flip_flop = _detect_flip_flop(statuses)

        alert_ids = [str(r.get("alert_id") or "") for r in rows_sorted if r.get("alert_id")]
        entity_records: list[dict[str, Any]] = []
        seen_entity_keys: set[str] = set()
        for alert_id in alert_ids:
            for entity_item in entity_map.get(alert_id, []):
                raw_key = _to_json(entity_item.get("entity_raw"))
                key = f"{entity_item.get('entity_name')}::{raw_key}"
                if key in seen_entity_keys:
                    continue
                seen_entity_keys.add(key)
                entity_records.append(entity_item)

        entity_names = sorted(
            {
                str(item.get("entity_name"))
                for item in entity_records
                if item.get("entity_name") not in (None, "")
            }
        )

        if not entity_names:
            fallback = str(current.get("affected_entities_detail") or "").strip()
            if fallback and fallback.lower() != "global":
                entity_names = [v.strip() for v in fallback.split(";") if v.strip()]

        affected_scope = str(current.get("affected_scope") or "").lower()
        if affected_scope == "global":
            affected_count = "global"
        else:
            affected_count = current.get("affected_entities_count")
            if affected_count in (None, "") and entity_names:
                affected_count = len(entity_names)

        result.append(
            {
                "account_id": current.get("account_id"),
                "account_name": current.get("account_name"),
                "integration_id": current.get("integration_id"),
                "integration_name": current.get("integration_name"),
                "integration_alias": current.get("integration_alias"),
                "saas_name": current.get("saas_name"),
                "security_check_id": current.get("security_check_id"),
                "security_check_name": current.get("security_check_name"),
                "security_check_api_link": current.get("security_check_api_link"),
                "check_key": build_check_key(
                    current.get("saas_name") or current.get("integration_name"),
                    current.get("integration_alias") or current.get("integration_name"),
                    current.get("security_check_name") or current.get("security_check_id"),
                ),
                "security_check_details": current.get("security_check_details"),
                "remediation_steps": current.get("remediation_steps"),
                "impact_level": current.get("impact_level"),
                "current_status": current_status,
                "change_datetime": current.get("change_datetime"),
                "affected_scope": affected_scope,
                "affected_count": affected_count,
                "entities": entity_names,
                "entity_records": entity_records,
                "flip_flop": flip_flop,
                "ticket_count_for_check": _safe_int(current.get("ticket_count_for_check"), default=0),
                "open_ticket_count_for_check": _safe_int(current.get("open_ticket_count_for_check"), default=0),
                "ticket_match_key": current.get("ticket_match_key"),
                "history": rows_sorted,
            }
        )

    result.sort(key=lambda g: (_status_rank(g["current_status"]),), reverse=False)
    return result


def render_configuration_drifts_ui(summary_df: pd.DataFrame, entities_df: pd.DataFrame) -> None:
    groups = build_drift_groups(summary_df, entities_df)
    if not groups:
        display(HTML('<div style="padding:10px;border:1px solid #ddd;border-radius:8px;">No configuration drift records found for the current window.</div>'))
        return

    grouped_by_status: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for group in groups:
        grouped_by_status[group["current_status"]].append(group)

    status_list = sorted(grouped_by_status.keys(), key=_status_rank)

    html_parts = [
        '<style>',
        '.as-ui {font-family: Arial, sans-serif; font-size: 13px; line-height: 1.4;}',
        '.as-title {font-size: 18px; font-weight: 700; margin-bottom: 10px;}',
        '.as-meta {color: #666; margin-bottom: 12px;}',
        '.as-status-group {border: 1px solid #ddd; border-radius: 10px; margin-bottom: 12px; background: #fff;}',
        '.as-status-summary {cursor: pointer; font-weight: 700; padding: 10px 12px; background: #f8f9fa; border-radius: 10px;}',
        '.as-group-body {padding: 10px 12px;}',
        '.as-timeline {position: relative; margin-left: 8px; padding-left: 24px;}',
        '.as-timeline::before {content: ""; position: absolute; left: 7px; top: 0; bottom: 0; border-left: 2px solid #e5e7eb;}',
        '.as-timeline-item {position: relative; margin-bottom: 12px;}',
        '.as-timeline-dot {position: absolute; left: -21px; top: 12px; width: 10px; height: 10px; border-radius: 999px; background: #9ca3af; border: 2px solid #fff; box-shadow: 0 0 0 1px #d1d5db;}',
        '.as-card {border: 1px solid #e5e7eb; border-radius: 8px; background: #fff;}',
        '.as-card.as-failed {border-left: 5px solid #dc2626;}',
        '.as-card.as-flipflop {border-right: 5px solid #f59e0b;}',
        '.as-card-summary {cursor: pointer; padding: 8px 10px; font-weight: 600; background: #fdfdfd;}',
        '.as-card-heading {color: #000; font-weight: 800;}',
        '.as-card-content {padding: 8px 10px 10px 10px;}',
        '.as-line {margin-bottom: 6px;}',
        '.as-key {font-weight: 600; color: #111827;}',
        '.as-val {color: #374151;}',
        '.as-muted {color: #6b7280;}',
        '.as-fold {margin-top: 6px; border: 1px dashed #d1d5db; border-radius: 6px; padding: 6px 8px; background: #fafafa;}',
        '.as-badge {display: inline-block; margin-left: 6px; padding: 2px 6px; border-radius: 999px; font-size: 11px;}',
        '.as-badge-status-failed {background: #fee2e2; color: #991b1b;}',
        '.as-badge-status-passed {background: #dcfce7; color: #166534;}',
        '.as-badge-status-other {background: #e5e7eb; color: #374151;}',
        '.as-badge-flip {background: #fef3c7; color: #92400e;}',
        '.as-history table {width: 100%; border-collapse: collapse;}',
        '.as-history th, .as-history td {border: 1px solid #e5e7eb; padding: 4px 6px; text-align: left;}',
        '.as-history th {background: #f9fafb;}',
        '.as-entities {margin-top: 6px;}',
        '.as-entities summary {color: #000; font-weight: 400;}',
        '.as-entity-preview {margin-top: 6px; color: #000; font-weight: 400; border: 1px solid #e5e7eb; border-radius: 6px; background: #fff; padding: 6px 8px;}',
        '.as-entity-preview .label {font-weight: 400; color: #000;}',
        '.as-entity-more {margin-top: 4px;}',
        '.as-entity-more summary {color: #000; font-weight: 400;}',
        '.as-json {margin: 6px 0 0 0; padding: 8px; background: #f3f4f6; border-radius: 6px; overflow-x: auto; white-space: pre-wrap;}',
        '</style>',
        '<div class="as-ui">',
        '<div class="as-title">Configuration Drift Timeline</div>',
        f'<div class="as-meta">Grouped checks: {len(groups)}. Failed groups are shown first. Latest events are shown on top within each status group.</div>',
    ]

    for status in status_list:
        raw_items = grouped_by_status[status]
        items = sorted(raw_items, key=lambda g: _parse_ts(g.get("change_datetime")), reverse=True)
        is_passed = status == "passed"
        open_attr = '' if is_passed else ' open'
        status_label = status.capitalize()

        html_parts.append(f'<details class="as-status-group"{open_attr}>')
        html_parts.append(f'<summary class="as-status-summary">{_escape(status_label)} ({len(items)})</summary>')
        html_parts.append('<div class="as-group-body">')
        html_parts.append('<div class="as-timeline">')

        for item in items:
            saas_display = item.get("saas_name") or item.get("integration_name") or "Unknown SaaS"
            integration_alias = item.get("integration_alias") or item.get("integration_name") or "Unknown alias"
            check_display = item.get("security_check_name") or item.get("security_check_id") or "Unknown check"

            status_kind = "failed" if item["current_status"] == "failed" else ("passed" if item["current_status"] == "passed" else "other")
            status_badge = _style_badge(item["current_status"].capitalize(), f"status-{status_kind}")
            flip_badge = _style_badge("FLIP-FLOP", "flip") if item["flip_flop"] else ""

            card_class = "as-card"
            if item["current_status"] == "failed":
                card_class += " as-failed"
            if item["flip_flop"]:
                card_class += " as-flipflop"

            entity_count = item["affected_count"] if item["affected_count"] not in (None, "") else len(item["entities"])
            entity_count_display = _format_count(entity_count, default="0")

            html_parts.append('<div class="as-timeline-item">')
            html_parts.append('<div class="as-timeline-dot"></div>')
            html_parts.append(f'<details class="{card_class}" open>')
            html_parts.append(
                '<summary class="as-card-summary">'
                f'<span class="as-card-heading">{_escape(saas_display)} | {_escape(integration_alias)} | {_escape(check_display)}</span> '
                f'{status_badge}{flip_badge} '
                f'<span class="as-muted">(affected: {_escape(entity_count_display)})</span>'
                '</summary>'
            )
            html_parts.append('<div class="as-card-content">')
            html_parts.append(f'<div class="as-line"><span class="as-key">Latest change:</span> <span class="as-val">{_escape(item.get("change_datetime") or "")}</span></div>')
            html_parts.append(f'<div class="as-line"><span class="as-key">Impact:</span> <span class="as-val">{_escape(item.get("impact_level") or "")}</span></div>')
            html_parts.append(f'<div class="as-line"><span class="as-key">Current status:</span> <span class="as-val">{_escape(item.get("current_status") or "")}</span></div>')

            html_parts.append('<details class="as-fold">')
            html_parts.append('<summary>Check details</summary>')
            html_parts.append(f'<div class="as-line"><span class="as-val">{_escape(item.get("security_check_details") or "")}</span></div>')
            html_parts.append('</details>')

            html_parts.append('<details class="as-fold">')
            html_parts.append('<summary>Remediation</summary>')
            html_parts.append(f'<div class="as-line"><span class="as-val">{_escape(item.get("remediation_steps") or "")}</span></div>')
            html_parts.append('</details>')

            html_parts.append('<details class="as-entities">')
            html_parts.append(f'<summary>Affected entities ({_escape(entity_count_display)})</summary>')
            if str(item.get("affected_scope") or "").lower() == "global":
                html_parts.append('<div class="as-line as-muted">Global setting (not entity-scoped).</div>')
            elif item["entity_records"]:
                for entity in item["entity_records"]:
                    preview = _entity_preview_model(entity)
                    name_text = str(preview.get("name") or "").strip()
                    emails = preview.get("emails") or []
                    dates = preview.get("dates") or []
                    html_parts.append('<div class="as-entity-preview">')
                    if name_text:
                        html_parts.append(f'<div><span class="label">Name:</span> <strong>{_escape(name_text)}</strong></div>')
                    if emails:
                        html_parts.append(f'<div><span class="label">Email:</span> {_escape(", ".join(emails))}</div>')
                    if dates:
                        html_parts.append(f'<div><span class="label">Date:</span> {_escape(" | ".join(dates))}</div>')
                    other_payload = _entity_other_payload(entity)
                    if other_payload:
                        html_parts.append('<details class="as-entity-more">')
                        html_parts.append('<summary>Other populated fields</summary>')
                        html_parts.append(_json_pretty_html(other_payload))
                        html_parts.append('</details>')
                    html_parts.append('</div>')
            else:
                html_parts.append('<div class="as-line as-muted">No entity list returned.</div>')
            html_parts.append('</details>')

            html_parts.append('<details class="as-history" style="margin-top:8px;">')
            html_parts.append(f'<summary>Timeline history ({len(item["history"])} events)</summary>')
            html_parts.append('<table><thead><tr><th>Datetime</th><th>Status</th><th>Alert Type</th><th>Affected</th></tr></thead><tbody>')
            for row in item["history"]:
                row_status = normalize_status(row.get("current_status")).capitalize()
                row_type = normalize_alert_type(row.get("alert_type"))
                row_affected = _format_count(row.get("affected_entities_count"), default="")
                html_parts.append(
                    '<tr>'
                    f'<td>{_escape(row.get("change_datetime") or "")}</td>'
                    f'<td>{_escape(row_status)}</td>'
                    f'<td>{_escape(row_type)}</td>'
                    f'<td>{_escape(row_affected)}</td>'
                    '</tr>'
                )
            html_parts.append('</tbody></table>')
            html_parts.append('</details>')

            html_parts.append('</div>')
            html_parts.append('</details>')
            html_parts.append('</div>')

        html_parts.append('</div>')
        html_parts.append('</div>')
        html_parts.append('</details>')

    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))


def _truncate_text(raw: Any, max_len: int = 160) -> str:
    text = _safe_text(raw)
    if len(text) <= max_len:
        return text
    return text[: max_len - 3] + "..."


def _snow_stale_class(days_ago: Any) -> str:
    value = _safe_int(days_ago, default=-1)
    if value >= 7:
        return "as-snow-stale-high"
    if value >= 3:
        return "as-snow-stale-mid"
    return "as-snow-stale-low"


def render_configuration_drifts_with_snow_ui(
    summary_df: pd.DataFrame,
    entities_df: pd.DataFrame,
    snow_ticket_details_df: pd.DataFrame | None,
    *,
    title: str = "Configuration Drift Timeline (Enhanced with ServiceNow)",
    empty_message: str = "No records found for the current window.",
) -> None:
    groups = build_drift_groups(summary_df, entities_df)
    if not groups:
        display(HTML(f'<div style="padding:10px;border:1px solid #ddd;border-radius:8px;">{_escape(empty_message)}</div>'))
        return

    ticket_map: dict[str, list[dict[str, Any]]] = defaultdict(list)
    if snow_ticket_details_df is not None and not snow_ticket_details_df.empty:
        for ticket in snow_ticket_details_df.to_dict("records"):
            key = _safe_text(ticket.get("ticket_match_key"))
            if key:
                ticket_map[key].append(ticket)

        for key, rows in ticket_map.items():
            rows.sort(
                key=lambda row: _parse_datetime_utc(row.get("ticket_last_update_datetime"))
                or _parse_datetime_utc(row.get("ticket_opened_datetime"))
                or datetime(1970, 1, 1, tzinfo=timezone.utc),
                reverse=True,
            )

    grouped_by_status: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for group in groups:
        grouped_by_status[group["current_status"]].append(group)

    status_list = sorted(grouped_by_status.keys(), key=_status_rank)

    html_parts = [
        '<style>',
        '.as-ui {font-family: Arial, sans-serif; font-size: 13px; line-height: 1.4;}',
        '.as-title {font-size: 18px; font-weight: 700; margin-bottom: 10px;}',
        '.as-meta {color: #666; margin-bottom: 12px;}',
        '.as-status-group {border: 1px solid #ddd; border-radius: 10px; margin-bottom: 12px; background: #fff;}',
        '.as-status-summary {cursor: pointer; font-weight: 700; padding: 10px 12px; background: #f8f9fa; border-radius: 10px;}',
        '.as-group-body {padding: 10px 12px;}',
        '.as-timeline {position: relative; margin-left: 8px; padding-left: 24px;}',
        '.as-timeline::before {content: ""; position: absolute; left: 7px; top: 0; bottom: 0; border-left: 2px solid #e5e7eb;}',
        '.as-timeline-item {position: relative; margin-bottom: 12px;}',
        '.as-timeline-dot {position: absolute; left: -21px; top: 12px; width: 10px; height: 10px; border-radius: 999px; background: #9ca3af; border: 2px solid #fff; box-shadow: 0 0 0 1px #d1d5db;}',
        '.as-card {border: 1px solid #e5e7eb; border-radius: 8px; background: #fff;}',
        '.as-card.as-failed {border-left: 5px solid #dc2626;}',
        '.as-card.as-flipflop {border-right: 5px solid #f59e0b;}',
        '.as-card-summary {cursor: pointer; padding: 8px 10px; font-weight: 600; background: #fdfdfd;}',
        '.as-card-heading {color: #000; font-weight: 800;}',
        '.as-card-content {padding: 8px 10px 10px 10px;}',
        '.as-line {margin-bottom: 6px;}',
        '.as-key {font-weight: 600; color: #111827;}',
        '.as-val {color: #374151;}',
        '.as-muted {color: #6b7280;}',
        '.as-fold {margin-top: 6px; border: 1px dashed #d1d5db; border-radius: 6px; padding: 6px 8px; background: #fafafa;}',
        '.as-badge {display: inline-block; margin-left: 6px; padding: 2px 6px; border-radius: 999px; font-size: 11px;}',
        '.as-badge-status-failed {background: #fee2e2; color: #991b1b;}',
        '.as-badge-status-passed {background: #dcfce7; color: #166534;}',
        '.as-badge-status-other {background: #e5e7eb; color: #374151;}',
        '.as-badge-flip {background: #fef3c7; color: #92400e;}',
        '.as-badge-snow-none {background: #e5e7eb; color: #374151;}',
        '.as-badge-snow-open {background: #fee2e2; color: #7f1d1d;}',
        '.as-badge-snow-closed {background: #dcfce7; color: #166534;}',
        '.as-snow-stale {display: inline-block; margin-left: 6px; font-size: 11px;}',
        '.as-snow-stale-low {color: #374151;}',
        '.as-snow-stale-mid {color: #b45309; font-weight: 600;}',
        '.as-snow-stale-high {color: #b91c1c; font-weight: 700;}',
        '.as-history table, .as-snow-table {width: 100%; border-collapse: collapse;}',
        '.as-history th, .as-history td, .as-snow-table th, .as-snow-table td {border: 1px solid #e5e7eb; padding: 4px 6px; text-align: left; vertical-align: top;}',
        '.as-history th, .as-snow-table th {background: #f9fafb;}',
        '.as-entities {margin-top: 6px;}',
        '.as-entities summary {color: #000; font-weight: 400;}',
        '.as-entity-preview {margin-top: 6px; color: #000; font-weight: 400; border: 1px solid #e5e7eb; border-radius: 6px; background: #fff; padding: 6px 8px;}',
        '.as-entity-preview .label {font-weight: 400; color: #000;}',
        '.as-entity-more {margin-top: 4px;}',
        '.as-entity-more summary {color: #000; font-weight: 400;}',
        '.as-json {margin: 6px 0 0 0; padding: 8px; background: #f3f4f6; border-radius: 6px; overflow-x: auto; white-space: pre-wrap;}',
        '</style>',
        '<div class="as-ui">',
        f'<div class="as-title">{_escape(title)}</div>',
        f'<div class="as-meta">Grouped checks: {len(groups)}. Failed groups are shown first. ServiceNow open tickets are highlighted.</div>',
    ]

    for status in status_list:
        raw_items = grouped_by_status[status]
        items = sorted(raw_items, key=lambda g: _parse_ts(g.get("change_datetime")), reverse=True)
        is_passed = status == "passed"
        open_attr = '' if is_passed else ' open'
        status_label = status.capitalize()

        html_parts.append(f'<details class="as-status-group"{open_attr}>')
        html_parts.append(f'<summary class="as-status-summary">{_escape(status_label)} ({len(items)})</summary>')
        html_parts.append('<div class="as-group-body">')
        html_parts.append('<div class="as-timeline">')

        for item in items:
            saas_display = item.get("saas_name") or item.get("integration_name") or "Unknown SaaS"
            integration_alias = item.get("integration_alias") or item.get("integration_name") or "Unknown alias"
            check_display = item.get("security_check_name") or item.get("security_check_id") or "Unknown check"

            status_kind = "failed" if item["current_status"] == "failed" else ("passed" if item["current_status"] == "passed" else "other")
            status_badge = _style_badge(item["current_status"].capitalize(), f"status-{status_kind}")
            flip_badge = _style_badge("FLIP-FLOP", "flip") if item["flip_flop"] else ""

            card_class = "as-card"
            if item["current_status"] == "failed":
                card_class += " as-failed"
            if item["flip_flop"]:
                card_class += " as-flipflop"

            entity_count = item["affected_count"] if item["affected_count"] not in (None, "") else len(item["entities"])
            entity_count_display = _format_count(entity_count, default="0")

            check_key = _safe_text(item.get("check_key") or item.get("ticket_match_key"))
            tickets = ticket_map.get(check_key, [])

            total_tickets = len(tickets)
            open_tickets = sum(1 for ticket in tickets if not bool(ticket.get("ticket_is_closed")))
            latest_ticket = tickets[0] if tickets else None
            as_check_link = _build_as_check_details_link(item)
            latest_ticket_link = _build_latest_snow_ticket_link(latest_ticket)

            snow_badge = '<span class="as-badge as-badge-snow-none">SN: none</span>'
            stale_hint = ""

            if total_tickets > 0:
                if open_tickets > 0:
                    snow_badge = f'<span class="as-badge as-badge-snow-open">SN: open {open_tickets}</span>'
                    latest_days = latest_ticket.get("ticket_last_update_days_ago") if latest_ticket else None
                    if latest_days not in (None, ""):
                        stale_class = _snow_stale_class(latest_days)
                        stale_hint = f'<span class="as-snow-stale {stale_class}">last update {_escape(_format_count(latest_days, default="0"))}d ago</span>'
                else:
                    snow_badge = f'<span class="as-badge as-badge-snow-closed">SN: closed {total_tickets}</span>'

            html_parts.append('<div class="as-timeline-item">')
            html_parts.append('<div class="as-timeline-dot"></div>')
            html_parts.append(f'<details class="{card_class}" open>')
            html_parts.append(
                '<summary class="as-card-summary">'
                f'<span class="as-card-heading">{_escape(saas_display)} | {_escape(integration_alias)} | {_escape(check_display)}</span> '
                f'{status_badge}{flip_badge}{snow_badge}{stale_hint} '
                f'<span class="as-muted">(affected: {_escape(entity_count_display)})</span>'
                '</summary>'
            )

            html_parts.append('<div class="as-card-content">')
            html_parts.append(f'<div class="as-line"><span class="as-key">Latest change:</span> <span class="as-val">{_escape(item.get("change_datetime") or "")}</span></div>')
            html_parts.append(f'<div class="as-line"><span class="as-key">Impact:</span> <span class="as-val">{_escape(item.get("impact_level") or "")}</span></div>')
            html_parts.append(f'<div class="as-line"><span class="as-key">Current status:</span> <span class="as-val">{_escape(item.get("current_status") or "")}</span></div>')
            link_parts = []
            check_link_html = _html_link("Adaptive Shield check details", as_check_link)
            if check_link_html:
                link_parts.append(check_link_html)
            sn_link_html = _html_link("Latest ServiceNow ticket", latest_ticket_link)
            if sn_link_html:
                link_parts.append(sn_link_html)
            if link_parts:
                html_parts.append(
                    '<div class="as-line"><span class="as-key">Links:</span> '
                    f'<span class="as-val">{" | ".join(link_parts)}</span></div>'
                )

            html_parts.append('<details class="as-fold">')
            html_parts.append('<summary>Check details</summary>')
            html_parts.append(f'<div class="as-line"><span class="as-val">{_escape(item.get("security_check_details") or "")}</span></div>')
            html_parts.append('</details>')

            html_parts.append('<details class="as-fold">')
            html_parts.append('<summary>Remediation</summary>')
            html_parts.append(f'<div class="as-line"><span class="as-val">{_escape(item.get("remediation_steps") or "")}</span></div>')
            html_parts.append('</details>')

            html_parts.append('<details class="as-fold">')
            html_parts.append(f'<summary>ServiceNow incidents ({total_tickets})</summary>')
            if tickets:
                html_parts.append('<table class="as-snow-table"><thead><tr><th>Number</th><th>Opened</th><th>State</th><th>Assigned to</th><th>Last update</th><th>Days ago</th><th>Note source</th><th>Last note</th></tr></thead><tbody>')
                for ticket in tickets:
                    note_text = _truncate_text(ticket.get("ticket_last_update_content"), max_len=140)
                    html_parts.append(
                        '<tr>'
                        f'<td>{_escape(ticket.get("ticket_number") or "")}</td>'
                        f'<td>{_escape(ticket.get("ticket_opened_datetime") or "")}</td>'
                        f'<td>{_escape(ticket.get("ticket_state") or "")}</td>'
                        f'<td>{_escape(ticket.get("ticket_assigned_to") or "")}</td>'
                        f'<td>{_escape(ticket.get("ticket_last_update_datetime") or "")}</td>'
                        f'<td>{_escape(_format_count(ticket.get("ticket_last_update_days_ago"), default=""))}</td>'
                        f'<td>{_escape(ticket.get("ticket_last_note_source") or "")}</td>'
                        f'<td>{_escape(note_text)}</td>'
                        '</tr>'
                    )
                html_parts.append('</tbody></table>')
            else:
                html_parts.append('<div class="as-line as-muted">No mapped incidents for this check.</div>')
            html_parts.append('</details>')

            html_parts.append('<details class="as-entities">')
            html_parts.append(f'<summary>Affected entities ({_escape(entity_count_display)})</summary>')
            if str(item.get("affected_scope") or "").lower() == "global":
                html_parts.append('<div class="as-line as-muted">Global setting (not entity-scoped).</div>')
            elif item["entity_records"]:
                for entity in item["entity_records"]:
                    preview = _entity_preview_model(entity)
                    name_text = str(preview.get("name") or "").strip()
                    emails = preview.get("emails") or []
                    dates = preview.get("dates") or []
                    html_parts.append('<div class="as-entity-preview">')
                    if name_text:
                        html_parts.append(f'<div><span class="label">Name:</span> <strong>{_escape(name_text)}</strong></div>')
                    if emails:
                        html_parts.append(f'<div><span class="label">Email:</span> {_escape(", ".join(emails))}</div>')
                    if dates:
                        html_parts.append(f'<div><span class="label">Date:</span> {_escape(" | ".join(dates))}</div>')
                    other_payload = _entity_other_payload(entity)
                    if other_payload:
                        html_parts.append('<details class="as-entity-more">')
                        html_parts.append('<summary>Other populated fields</summary>')
                        html_parts.append(_json_pretty_html(other_payload))
                        html_parts.append('</details>')
                    html_parts.append('</div>')
            else:
                html_parts.append('<div class="as-line as-muted">No entity list returned.</div>')
            html_parts.append('</details>')

            html_parts.append('<details class="as-history" style="margin-top:8px;">')
            html_parts.append(f'<summary>Timeline history ({len(item["history"])} events)</summary>')
            html_parts.append('<table><thead><tr><th>Datetime</th><th>Status</th><th>Alert Type</th><th>Affected</th></tr></thead><tbody>')
            for row in item["history"]:
                row_status = normalize_status(row.get("current_status")).capitalize()
                row_type = normalize_alert_type(row.get("alert_type"))
                row_affected = _format_count(row.get("affected_entities_count"), default="")
                html_parts.append(
                    '<tr>'
                    f'<td>{_escape(row.get("change_datetime") or "")}</td>'
                    f'<td>{_escape(row_status)}</td>'
                    f'<td>{_escape(row_type)}</td>'
                    f'<td>{_escape(row_affected)}</td>'
                    '</tr>'
                )
            html_parts.append('</tbody></table>')
            html_parts.append('</details>')

            html_parts.append('</div>')
            html_parts.append('</details>')
            html_parts.append('</div>')

        html_parts.append('</div>')
        html_parts.append('</div>')
        html_parts.append('</details>')

    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))


def render_alert_types_with_snow_ui(
    summary_df: pd.DataFrame,
    entities_df: pd.DataFrame,
    snow_ticket_details_df: pd.DataFrame | None,
) -> None:
    if summary_df is None or summary_df.empty:
        display(HTML('<div style="padding:10px;border:1px solid #ddd;border-radius:8px;">No alerts found for the current window.</div>'))
        return

    if "alert_type" not in summary_df.columns:
        render_configuration_drifts_with_snow_ui(
            summary_df.copy(),
            entities_df,
            snow_ticket_details_df,
            title="Alerts Timeline (Enhanced with ServiceNow)",
            empty_message="No alerts found for the current window.",
        )
        return

    working_df = summary_df.copy()
    working_df["_alert_type_norm"] = (
        working_df["alert_type"].fillna("").astype(str).map(normalize_alert_type).replace("", "unknown")
    )

    seen_types = sorted(
        set(value for value in working_df["_alert_type_norm"].dropna().astype(str).tolist() if value)
    )
    ordered_types: list[str] = []
    for fixed in ("configuration_drift", "integration_failure"):
        if fixed in seen_types:
            ordered_types.append(fixed)
            seen_types.remove(fixed)
    ordered_types.extend(seen_types)

    for alert_type in ordered_types:
        subset = working_df[working_df["_alert_type_norm"] == alert_type].drop(columns=["_alert_type_norm"], errors="ignore")
        if subset.empty:
            continue

        if alert_type == "configuration_drift":
            section_label = "Configuration Drift"
        elif alert_type == "integration_failure":
            section_label = "Integration Failure"
        else:
            section_label = f"other:{alert_type}"

        render_configuration_drifts_with_snow_ui(
            subset,
            entities_df,
            snow_ticket_details_df,
            title=f"{section_label} Timeline (Enhanced with ServiceNow)",
            empty_message=f"No records found for section {section_label}.",
        )


def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / ".env.example").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return cwd


def _render_initialization_status(config: dict[str, Any]) -> None:
    rows = [
        {
            "name": "AS_API_KEY",
            "required": True,
            "ready": bool(_safe_text(config.get("as_api_key"))),
            "display": "***" if bool(_safe_text(config.get("as_api_key"))) else "",
        },
        {
            "name": "SNOW_BASE_URL",
            "required": bool(config.get("snow_enabled")),
            "ready": bool(_safe_text(config.get("snow_base_url"))),
            "display": _safe_text(config.get("snow_base_url")),
        },
        {
            "name": "LOOKBACK_DAYS",
            "required": True,
            "ready": True,
            "display": _safe_text(config.get("lookback_days")),
        },
        {
            "name": "SNOW_BROWSER",
            "required": False,
            "ready": True,
            "display": _safe_text(config.get("snow_browser")),
        },
        {
            "name": "CACHE_ROOT",
            "required": False,
            "ready": True,
            "display": _safe_text(config.get("cache_root")),
        },
    ]

    html_rows = []
    for row in rows:
        required_text = "yes" if row["required"] else "no"
        status_text = "ready" if row["ready"] else "missing"
        status_color = "#166534" if row["ready"] else "#b91c1c"
        html_rows.append(
            "<tr>"
            f"<td>{_escape(row['name'])}</td>"
            f"<td>{_escape(required_text)}</td>"
            f"<td style='color:{status_color};font-weight:600'>{_escape(status_text)}</td>"
            f"<td>{_escape(row['display'])}</td>"
            "</tr>"
        )

    display(
        HTML(
            "<div style='margin:8px 0 12px 0;'>"
            "<div style='font-weight:700;margin-bottom:6px;'>Initialization readiness</div>"
            "<table style='border-collapse:collapse;width:100%;font-size:12px;'>"
            "<thead><tr>"
            "<th style='text-align:left;border:1px solid #ddd;padding:4px 6px;'>Variable</th>"
            "<th style='text-align:left;border:1px solid #ddd;padding:4px 6px;'>Required</th>"
            "<th style='text-align:left;border:1px solid #ddd;padding:4px 6px;'>Status</th>"
            "<th style='text-align:left;border:1px solid #ddd;padding:4px 6px;'>Value</th>"
            "</tr></thead>"
            f"<tbody>{''.join(html_rows)}</tbody>"
            "</table>"
            "</div>"
        )
    )


def _init_cache_toggle_ui(config: dict[str, Any], runtime_cache_flags: dict[str, bool]) -> None:
    adaptive_shield_enabled = _cache_group_enabled(
        runtime_cache_flags,
        AS_CACHE_DATASET_KEYS,
        default=bool(config.get("use_cache_adaptive_shield", True)),
    )
    service_now_enabled = _cache_group_enabled(
        runtime_cache_flags,
        SNOW_CACHE_DATASET_KEYS,
        default=bool(config.get("use_cache_service_now", True)),
    )

    if widgets is None:
        print("ipywidgets not installed; cache toggles fallback to config/cache env values.")
        print(
            "Cache toggles:",
            {
                "use_cache_adaptive_shield": adaptive_shield_enabled,
                "use_cache_service_now": service_now_enabled,
            },
        )
        return

    adaptive_shield_checkbox = widgets.Checkbox(
        value=adaptive_shield_enabled,
        description="Use cache for Adaptive Shield",
        indent=False,
    )
    service_now_checkbox = widgets.Checkbox(
        value=service_now_enabled,
        description="Use cache for ServiceNow",
        indent=False,
    )

    apply_button = widgets.Button(description="Apply Cache Settings", button_style="primary")
    output = widgets.Output()

    def _on_apply(_: Any) -> None:
        config["use_cache_adaptive_shield"] = bool(adaptive_shield_checkbox.value)
        config["use_cache_service_now"] = bool(service_now_checkbox.value)

        selected_flags = _build_runtime_cache_flags(
            use_cache_adaptive_shield=config["use_cache_adaptive_shield"],
            use_cache_service_now=config["use_cache_service_now"],
        )

        runtime_cache_flags.clear()
        runtime_cache_flags.update(selected_flags)
        config["cache_flags"] = runtime_cache_flags

        with output:
            clear_output(wait=True)
            print(
                "Applied cache settings:",
                {
                    "use_cache_adaptive_shield": config["use_cache_adaptive_shield"],
                    "use_cache_service_now": config["use_cache_service_now"],
                },
            )

    apply_button.on_click(_on_apply)
    display(
        widgets.VBox(
            [
                widgets.HTML("<b>Cache toggles (unchecked = disable read + write)</b>"),
                adaptive_shield_checkbox,
                service_now_checkbox,
                apply_button,
                output,
            ]
        )
    )

ROOT_DIR = resolve_project_root()
load_dotenv(ROOT_DIR / ".env")

RUN_TS = datetime.now(timezone.utc)
LOOKBACK_DAYS = int(os.getenv("LOOKBACK_DAYS", "3"))
FROM_DATE = (RUN_TS - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")
TO_DATE = RUN_TS.strftime("%Y-%m-%dT%H:%M:%SZ")

config = {
    "as_api_key": os.getenv("AS_API_KEY", "").strip(),
    "as_base_url": os.getenv("AS_BASE_URL", "https://api.adaptive-shield.com").strip(),
    "as_account_ids": [
        account.strip()
        for account in os.getenv("AS_ACCOUNT_IDS", "").split(",")
        if account.strip()
    ],
    "lookback_days": LOOKBACK_DAYS,
    "from_date": FROM_DATE,
    "to_date": TO_DATE,
    "output_root": os.getenv("OUTPUT_ROOT", "output").strip(),
    "cache_root": os.getenv("CACHE_ROOT", str(ROOT_DIR / ".cache" / "as_weekly_report")).strip(),
    "use_cache_adaptive_shield": env_bool("USE_CACHE_ADAPTIVE_SHIELD", env_bool("CACHE_AS", True)),
    "use_cache_service_now": env_bool("USE_CACHE_SERVICE_NOW", env_bool("CACHE_SNOW", True)),
    "export_xlsx": env_bool("EXPORT_XLSX", True),
    "export_csv": env_bool("EXPORT_CSV", True),
    "snow_enabled": env_bool("SNOW_ENABLED", False),
    "snow_base_url": os.getenv("SNOW_BASE_URL", "").strip(),
    "snow_username": os.getenv("SNOW_USERNAME", "").strip(),
    "snow_password": os.getenv("SNOW_PASSWORD", "").strip(),
    "snow_cookie_path": os.getenv("SNOW_COOKIE_PATH", str(ROOT_DIR / ".snow_cookies.pkl")).strip(),
    "snow_browser": os.getenv("SNOW_BROWSER", "edge").strip().lower(),
    "snow_edge_profile": os.getenv("SNOW_EDGE_PROFILE", "Default").strip(),
    "snow_keep_browser_open": env_bool("SNOW_KEEP_BROWSER_OPEN", False),
    "snow_headless": env_bool("SNOW_HEADLESS", False),
    "snow_user_data_dir": os.getenv("SNOW_USER_DATA_DIR", "").strip(),
    "snow_chromedriver_path": os.getenv("SNOW_CHROMEDRIVER_PATH", "").strip(),
    "snow_login_timeout_seconds": int(os.getenv("SNOW_LOGIN_TIMEOUT_SECONDS", "20")),
    "snow_max_incidents": int(os.getenv("SNOW_MAX_INCIDENTS", "200")),
    "snow_fetch_detail_notes": env_bool("SNOW_FETCH_DETAIL_NOTES", True),
    "snow_interactive_login": env_bool("SNOW_INTERACTIVE_LOGIN", True),
    "snow_manual_login_timeout_seconds": int(os.getenv("SNOW_MANUAL_LOGIN_TIMEOUT_SECONDS", "300")),
    "snow_manual_login_poll_seconds": int(os.getenv("SNOW_MANUAL_LOGIN_POLL_SECONDS", "3")),
    "snow_open_baseurl_first": env_bool("SNOW_OPEN_BASEURL_FIRST", True),
    "snow_use_table_api": env_bool("SNOW_USE_TABLE_API", True),
    "snow_table_api_only": env_bool("SNOW_TABLE_API_ONLY", False),
    "snow_incident_query": os.getenv("SNOW_INCIDENT_QUERY", "short_descriptionLIKEAdaptiveShield").strip(),
    "snow_date_filter_enabled": env_bool("SNOW_DATE_FILTER_ENABLED", True),
    "snow_date_filter_field": os.getenv("SNOW_DATE_FILTER_FIELD", "opened_at").strip() or "opened_at",
    "rate_limit_per_minute": int(os.getenv("RATE_LIMIT_PER_MINUTE", "90")),
    "request_timeout_seconds": int(os.getenv("REQUEST_TIMEOUT_SECONDS", "30")),
    "max_retries": int(os.getenv("MAX_RETRIES", "3")),
}

RUN_DAY_DIR = ROOT_DIR / config["output_root"] / RUN_TS.strftime("%Y-%m-%d")
RUN_DAY_DIR.mkdir(parents=True, exist_ok=True)

runtime_cache_flags: dict[str, bool] = _build_runtime_cache_flags(
    use_cache_adaptive_shield=bool(config.get("use_cache_adaptive_shield", True)),
    use_cache_service_now=bool(config.get("use_cache_service_now", True)),
)
config["cache_flags"] = runtime_cache_flags
config["cache_store"] = WebCacheStore(config["cache_root"], RUN_DAY_DIR)

pipeline_errors = []

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
print("Configuration loaded:")
redacted_config = {
    k: v
    for k, v in config.items()
    if k not in {"as_api_key", "snow_password", "cache_store"}
}
if config.get("snow_password"):
    redacted_config["snow_password"] = "***"
print(redacted_config)
print("Project root:", ROOT_DIR)
print("Output directory:", RUN_DAY_DIR)

_render_initialization_status(config)
_init_cache_toggle_ui(config, runtime_cache_flags)

if not config["as_api_key"]:
    raise ValueError("AS_API_KEY is required. Add it to .env before running.")


In [ ]:
# Cell 2: Unified Data Fetching + Processing (with stage progress)

STAGE_DEFS = [
    ("snow_prefetch", "ServiceNow pre-fetch"),
    ("client_init", "API client initialization"),
    ("accounts", "Get accounts"),
    ("alerts", "Fetch alerts"),
    ("security_checks", "Enrich security checks"),
    ("affected_entities", "Enrich affected entities"),
    ("summary", "Build summary table"),
    ("snow_mapping", "ServiceNow mapping"),
    ("quality", "Data quality checks"),
]

stage_progress_widgets: dict[str, dict[str, Any]] = {}


def _init_stage_progress(stages: list[tuple[str, str]]) -> None:
    if widgets is None:
        print("ipywidgets not installed; stage progress fallback to text logs.")
        return

    rows = []
    for stage_key, stage_label in stages:
        progress = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description=stage_label,
            bar_style="",
            style={"description_width": "190px"},
            layout=widgets.Layout(width="460px"),
        )
        status = widgets.HTML(value="<span style='font-size:12px;color:#6b7280'>pending</span>")
        rows.append(widgets.VBox([progress, status]))
        stage_progress_widgets[stage_key] = {"progress": progress, "status": status}

    display(widgets.VBox(rows))


def _set_stage_progress(stage_key: str, value: int, message: str = "", *, style: str = "") -> None:
    progress_value = max(0, min(100, int(value)))

    if widgets is None or stage_key not in stage_progress_widgets:
        if message:
            print(f"[{stage_key}] {progress_value}% {message}")
        return

    item = stage_progress_widgets[stage_key]
    item["progress"].value = progress_value
    item["progress"].bar_style = style

    color = "#334155"
    if style == "success":
        color = "#166534"
    elif style == "danger":
        color = "#b91c1c"
    elif style == "warning":
        color = "#92400e"

    item["status"].value = f"<span style='font-size:12px;color:{color}'>{_escape(message)}</span>"


_init_stage_progress(STAGE_DEFS)

# Stage 1: ServiceNow pre-fetch
_set_stage_progress("snow_prefetch", 5, "Initializing ServiceNow pre-fetch...")
snow_prefetch_attempted = False
snow_prefetched_incidents = []
snow_prefetch_raw_df = pd.DataFrame(columns=SNOW_RAW_COLUMNS)

try:
    if config["snow_enabled"]:
        snow_prefetch_attempted = True

        if config.get("snow_interactive_login", True) and config.get("snow_headless", False):
            print("SNOW_INTERACTIVE_LOGIN=true detected, forcing SNOW_HEADLESS=false for visible SSO login.")
            config["snow_headless"] = False

        _set_stage_progress("snow_prefetch", 35, "Collecting incidents from ServiceNow...")
        snow_prefetched_incidents = _collect_snow_incidents(
            config=config,
            lookback_days=config["lookback_days"],
            run_ts_utc=RUN_TS,
            pipeline_errors=pipeline_errors,
        )

        if snow_prefetched_incidents:
            snow_prefetch_raw_df = pd.DataFrame(snow_prefetched_incidents)
            snow_prefetch_raw_df = snow_prefetch_raw_df.reindex(columns=SNOW_RAW_COLUMNS, fill_value=None)

        print("ServiceNow pre-fetch incidents:", len(snow_prefetched_incidents))
        display(snow_prefetch_raw_df.head(20))
        _set_stage_progress("snow_prefetch", 100, f"Done ({len(snow_prefetched_incidents)} incidents).", style="success")
    else:
        print("ServiceNow pre-fetch skipped (SNOW_ENABLED=false).")
        _set_stage_progress("snow_prefetch", 100, "Skipped (SNOW_ENABLED=false).", style="success")
except Exception as exc:
    _set_stage_progress("snow_prefetch", 100, f"Failed: {exc}", style="danger")
    raise

# Stage 2: API client initialization
_set_stage_progress("client_init", 10, "Initializing AdaptiveShieldClient...")
try:
    client = AdaptiveShieldClient(
        api_key=config["as_api_key"],
        base_url=config["as_base_url"],
        rate_limit_per_minute=config["rate_limit_per_minute"],
        timeout_seconds=config["request_timeout_seconds"],
        max_retries=config["max_retries"],
        cache_store=config.get("cache_store"),
        cache_flags=config.get("cache_flags"),
    )

    print("AdaptiveShieldClient initialized")
    _set_stage_progress("client_init", 100, "Done.", style="success")
except Exception as exc:
    _set_stage_progress("client_init", 100, f"Failed: {exc}", style="danger")
    raise

# Stage 3: Get accounts
_set_stage_progress("accounts", 5, "Loading account list...")
accounts_records = []

try:
    if config["as_account_ids"]:
        total_ids = len(config["as_account_ids"])
        for idx, account_id in enumerate(config["as_account_ids"], start=1):
            accounts_records.append({"id": account_id, "name": ""})
            progress = 5 + int((idx / max(1, total_ids)) * 90)
            _set_stage_progress("accounts", progress, f"Using configured account IDs ({idx}/{total_ids})...")
    else:
        _set_stage_progress("accounts", 40, "Fetching accounts from API...")
        accounts_records = client.get_accounts()

    accounts_df = pd.DataFrame(accounts_records)
    if not accounts_df.empty:
        accounts_df = accounts_df.rename(columns={"id": "account_id", "name": "account_name"})
        for column in ["account_id", "account_name"]:
            if column not in accounts_df.columns:
                accounts_df[column] = ""
        accounts_df = accounts_df[["account_id", "account_name"]]
    else:
        accounts_df = pd.DataFrame(columns=["account_id", "account_name"])

    display(accounts_df.head(20))
    print("Accounts found:", len(accounts_df))
    _set_stage_progress("accounts", 100, f"Done ({len(accounts_df)} accounts).", style="success")
except Exception as exc:
    _set_stage_progress("accounts", 100, f"Failed: {exc}", style="danger")
    raise

# Stage 4: Fetch alerts
_set_stage_progress("alerts", 5, "Fetching alerts by account and type...")
TARGET_ALERT_TYPES = ["configuration_drift", "integration_failure"]

alerts_raw_rows = []

try:
    account_ids = accounts_df["account_id"].dropna().astype(str).tolist()
    total_tasks = max(1, len(account_ids) * len(TARGET_ALERT_TYPES))
    completed_tasks = 0

    for account_idx, account_id in enumerate(account_ids, start=1):
        for alert_type in TARGET_ALERT_TYPES:
            try:
                rows = client.get_alerts(
                    account_id=account_id,
                    from_date=config["from_date"],
                    to_date=config["to_date"],
                    alert_type=alert_type,
                )
                for row in rows:
                    row.setdefault("account_id", account_id)
                alerts_raw_rows.extend(rows)
            except Exception as exc:
                pipeline_errors.append(
                    {
                        "stage": "alerts",
                        "account_id": account_id,
                        "security_check_id": None,
                        "alert_id": None,
                        "message": str(exc),
                    }
                )

            completed_tasks += 1
            progress = 5 + int((completed_tasks / total_tasks) * 90)
            _set_stage_progress(
                "alerts",
                progress,
                f"Account {account_idx}/{max(1, len(account_ids))} · {alert_type}",
            )

    alerts_raw_df = pd.DataFrame(alerts_raw_rows)
    if not alerts_raw_df.empty and "id" in alerts_raw_df.columns:
        alerts_raw_df = alerts_raw_df.drop_duplicates(subset=["account_id", "id"], keep="first")

    alerts_filtered_rows = filter_target_alerts(alerts_raw_df.to_dict("records"), include_check_degraded=False)
    alerts_filtered_df = pd.DataFrame(alerts_filtered_rows)

    display(alerts_filtered_df.head(20))
    print("Raw alerts:", len(alerts_raw_df), "| Filtered alerts:", len(alerts_filtered_df))
    _set_stage_progress(
        "alerts",
        100,
        f"Done ({len(alerts_filtered_df)} filtered / {len(alerts_raw_df)} raw).",
        style="success",
    )
except Exception as exc:
    _set_stage_progress("alerts", 100, f"Failed: {exc}", style="danger")
    raise

# Stage 5: Enrich security checks
_set_stage_progress("security_checks", 5, "Enriching security check details...")
check_cache = {}
check_rows = []
enable_check_cache = bool(config.get("cache_flags", {}).get("as_security_checks", True))

try:
    alert_records = alerts_filtered_df.to_dict("records")
    total_alerts = max(1, len(alert_records))

    for idx, alert in enumerate(alert_records, start=1):
        alert_id = str(alert.get("id") or "")
        account_id = str(alert.get("account_id") or "")
        security_check_id = extract_security_check_id(alert)

        if not account_id or not security_check_id:
            pipeline_errors.append(
                {
                    "stage": "security_check",
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "alert_id": alert_id,
                    "message": "Missing account_id or security_check_id",
                }
            )
            check_rows.append(
                {
                    "alert_id": alert_id,
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                }
            )
            progress = 5 + int((idx / total_alerts) * 90)
            _set_stage_progress("security_checks", progress, f"Processed {idx}/{len(alert_records)} alerts")
            continue

        cache_key = (account_id, security_check_id)
        check_payload: dict[str, Any] = {}
        cache_miss = (not enable_check_cache) or (cache_key not in check_cache)
        if cache_miss:
            try:
                check_payload = client.get_security_check(account_id, security_check_id)
                if enable_check_cache:
                    check_cache[cache_key] = check_payload
            except Exception as exc:
                if enable_check_cache:
                    check_cache[cache_key] = {}
                pipeline_errors.append(
                    {
                        "stage": "security_check",
                        "account_id": account_id,
                        "security_check_id": security_check_id,
                        "alert_id": alert_id,
                        "message": str(exc),
                    }
                )

        if enable_check_cache and cache_key in check_cache:
            check_data = dict(check_cache[cache_key])
        else:
            check_data = dict(check_payload)

        check_rows.append(
            {
                "alert_id": alert_id,
                "account_id": account_id,
                "security_check_id": security_check_id,
                **check_data,
            }
        )

        progress = 5 + int((idx / total_alerts) * 90)
        _set_stage_progress("security_checks", progress, f"Processed {idx}/{len(alert_records)} alerts")

    checks_df = pd.DataFrame(check_rows)
    display(checks_df.head(20))
    print("Security checks enriched:", len(checks_df))
    _set_stage_progress("security_checks", 100, f"Done ({len(checks_df)} rows).", style="success")
except Exception as exc:
    _set_stage_progress("security_checks", 100, f"Failed: {exc}", style="danger")
    raise

# Stage 6: Enrich affected entities
_set_stage_progress("affected_entities", 5, "Resolving affected entities...")
entity_records = []
affected_resolve_log_rows = []
affected_cache = {}
enable_affected_cache = bool(config.get("cache_flags", {}).get("as_affected_entities", True))

try:
    alert_records = alerts_filtered_df.to_dict("records")
    total_alerts = max(1, len(alert_records))

    for idx, alert in enumerate(alert_records, start=1):
        alert_id = str(alert.get("id") or "")
        account_id = str(alert.get("account_id") or "")
        security_check_id = extract_security_check_id(alert)

        if "alert_id" in checks_df.columns:
            check_row = checks_df[checks_df["alert_id"].astype(str) == alert_id]
            check_obj = check_row.iloc[0].to_dict() if not check_row.empty else {}
        else:
            check_obj = {}
        is_global = bool(check_obj.get("is_global"))

        if is_global:
            affected_resolve_log_rows.append(
                {
                    "alert_id": alert_id,
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "resolve_status": "global",
                    "message": "Security check marked as global",
                }
            )
            progress = 5 + int((idx / total_alerts) * 90)
            _set_stage_progress("affected_entities", progress, f"Processed {idx}/{len(alert_records)} alerts")
            continue

        if not security_check_id:
            pipeline_errors.append(
                {
                    "stage": "affected_entities",
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "alert_id": alert_id,
                    "message": "Missing security_check_id",
                }
            )
            affected_resolve_log_rows.append(
                {
                    "alert_id": alert_id,
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "resolve_status": "unresolved",
                    "message": "Missing security_check_id",
                }
            )
            progress = 5 + int((idx / total_alerts) * 90)
            _set_stage_progress("affected_entities", progress, f"Processed {idx}/{len(alert_records)} alerts")
            continue

        cache_key = (account_id, security_check_id)
        entities = []
        cache_miss = (not enable_affected_cache) or (cache_key not in affected_cache)
        if cache_miss:
            try:
                fetched_entities = client.get_affected_entities(account_id, security_check_id)
                if isinstance(fetched_entities, list):
                    entities = [item for item in fetched_entities if isinstance(item, dict)]
                if enable_affected_cache:
                    affected_cache[cache_key] = entities
            except Exception as exc:
                if enable_affected_cache:
                    affected_cache[cache_key] = []
                pipeline_errors.append(
                    {
                        "stage": "affected_entities",
                        "account_id": account_id,
                        "security_check_id": security_check_id,
                        "alert_id": alert_id,
                        "message": str(exc),
                    }
                )
                affected_resolve_log_rows.append(
                    {
                        "alert_id": alert_id,
                        "account_id": account_id,
                        "security_check_id": security_check_id,
                        "resolve_status": "unresolved",
                        "message": "Affected API failed",
                    }
                )
                progress = 5 + int((idx / total_alerts) * 90)
                _set_stage_progress("affected_entities", progress, f"Processed {idx}/{len(alert_records)} alerts")
                continue

        if enable_affected_cache and cache_key in affected_cache:
            entities = [item for item in affected_cache[cache_key] if isinstance(item, dict)]

        for entity in entities:
            entity_records.append(
                {
                    "alert_id": alert_id,
                    "account_id": account_id,
                    "security_check_id": security_check_id,
                    "entity": entity,
                }
            )

        affected_resolve_log_rows.append(
            {
                "alert_id": alert_id,
                "account_id": account_id,
                "security_check_id": security_check_id,
                "resolve_status": "fetched",
                "message": f"Fetched {len(entities)} entities",
            }
        )

        progress = 5 + int((idx / total_alerts) * 90)
        _set_stage_progress("affected_entities", progress, f"Processed {idx}/{len(alert_records)} alerts")

    entities_df = build_entities_df(entity_records)
    affected_resolve_log_df = pd.DataFrame(affected_resolve_log_rows)

    display(entities_df.head(20))
    display(affected_resolve_log_df.head(20))
    print("Entity rows:", len(entities_df))
    _set_stage_progress("affected_entities", 100, f"Done ({len(entities_df)} entity rows).", style="success")
except Exception as exc:
    _set_stage_progress("affected_entities", 100, f"Failed: {exc}", style="danger")
    raise

# Stage 7: Build summary table
_set_stage_progress("summary", 20, "Building summary table...")
try:
    summary_df = build_summary_df(
        alerts=alerts_filtered_df.to_dict("records"),
        checks=checks_df.to_dict("records"),
        entities=entities_df.to_dict("records"),
        accounts=accounts_df.rename(columns={"account_id": "id", "account_name": "name"}).to_dict("records"),
        extracted_at_utc=RUN_TS.strftime("%Y-%m-%dT%H:%M:%SZ"),
        as_base_url=config.get("as_base_url", "https://api.adaptive-shield.com"),
    )

    display(summary_df.head(20))
    print("Summary rows:", len(summary_df))
    _set_stage_progress("summary", 100, f"Done ({len(summary_df)} rows).", style="success")
except Exception as exc:
    _set_stage_progress("summary", 100, f"Failed: {exc}", style="danger")
    raise

# Stage 8: ServiceNow mapping
_set_stage_progress("snow_mapping", 10, "Mapping ServiceNow incidents to summary...")
try:
    if "snow_prefetch_attempted" not in globals():
        snow_prefetch_attempted = False
    if "snow_prefetched_incidents" not in globals():
        snow_prefetched_incidents = []

    if config["snow_enabled"]:
        if snow_prefetch_attempted:
            _set_stage_progress("snow_mapping", 40, "Using pre-fetched ServiceNow incidents...")
            snow_row_df, snow_incidents_raw_df, snow_ticket_details_df = _map_incidents_to_summary(
                summary_df,
                snow_prefetched_incidents,
                run_ts_utc=RUN_TS,
            )
        else:
            _set_stage_progress("snow_mapping", 40, "Fetching related ServiceNow tickets...")
            snow_row_df, snow_incidents_raw_df, snow_ticket_details_df = fetch_related_tickets(
                summary_df,
                config["lookback_days"],
                config=config,
                run_ts_utc=RUN_TS,
                pipeline_errors=pipeline_errors,
            )
    else:
        snow_row_df, snow_incidents_raw_df, snow_ticket_details_df = _empty_snow_outputs()

    summary_with_snow_df = merge_snow_columns(summary_df, snow_row_df)

    print("ServiceNow row-mapped records:", len(snow_row_df))
    print("ServiceNow raw incidents:", len(snow_incidents_raw_df))
    print("ServiceNow ticket details:", len(snow_ticket_details_df))

    display(snow_incidents_raw_df.head(20))
    display(snow_ticket_details_df.head(20))
    display(summary_with_snow_df.head(20))
    _set_stage_progress("snow_mapping", 100, f"Done ({len(snow_ticket_details_df)} detail rows).", style="success")
except Exception as exc:
    _set_stage_progress("snow_mapping", 100, f"Failed: {exc}", style="danger")
    raise

# Stage 9: Data quality checks
_set_stage_progress("quality", 10, "Running data quality checks...")
try:
    quality_rows = []

    missing_summary_columns = [col for col in SUMMARY_COLUMNS if col not in summary_with_snow_df.columns]
    missing_entity_columns = [col for col in ENTITY_COLUMNS if col not in entities_df.columns]
    missing_snow_columns = [col for col in SNOW_COLUMNS if col not in summary_with_snow_df.columns]

    quality_rows.append(
        {
            "check": "missing_summary_columns",
            "value": len(missing_summary_columns),
            "details": ", ".join(missing_summary_columns),
        }
    )
    quality_rows.append(
        {
            "check": "missing_entity_columns",
            "value": len(missing_entity_columns),
            "details": ", ".join(missing_entity_columns),
        }
    )
    quality_rows.append(
        {
            "check": "missing_snow_columns",
            "value": len(missing_snow_columns),
            "details": ", ".join(missing_snow_columns),
        }
    )

    if not summary_with_snow_df.empty:
        duplicate_alert_keys = summary_with_snow_df.duplicated(subset=["account_id", "alert_id"]).sum()
    else:
        duplicate_alert_keys = 0

    quality_rows.append(
        {
            "check": "duplicate_account_alert_keys",
            "value": int(duplicate_alert_keys),
            "details": "subset=[account_id, alert_id]",
        }
    )

    for column in ["change_datetime", "security_check_name", "impact_level", "current_status", "ticket_number"]:
        if column in summary_with_snow_df.columns and not summary_with_snow_df.empty:
            null_ratio = float(summary_with_snow_df[column].isna().mean())
        else:
            null_ratio = 1.0
        quality_rows.append(
            {
                "check": f"null_ratio::{column}",
                "value": round(null_ratio, 4),
                "details": "fraction of null values",
            }
        )

    quality_rows.append(
        {
            "check": "snow_raw_incidents_rows",
            "value": int(len(snow_incidents_raw_df)),
            "details": "total incidents fetched from ServiceNow list",
        }
    )
    quality_rows.append(
        {
            "check": "snow_mapped_ticket_rows",
            "value": int(len(snow_ticket_details_df)),
            "details": "rows mapped to checks",
        }
    )

    quality_report_df = pd.DataFrame(quality_rows)
    errors_df = pd.DataFrame(pipeline_errors)

    display(quality_report_df)
    display(errors_df.head(20))
    print("Pipeline errors:", len(errors_df))
    _set_stage_progress("quality", 100, f"Done ({len(quality_rows)} checks, {len(errors_df)} errors).", style="success")
except Exception as exc:
    _set_stage_progress("quality", 100, f"Failed: {exc}", style="danger")
    raise


In [ ]:
# Cell 3: Alerts UI (ServiceNow toggle)

def _render_alerts_ui_without_snow() -> None:
    if "alert_type" in summary_df.columns:
        config_drift_df = summary_df[
            summary_df["alert_type"].fillna("").astype(str).str.lower() == "configuration_drift"
        ].copy()
    else:
        config_drift_df = summary_df.copy()

    render_configuration_drifts_ui(config_drift_df, entities_df)


def _render_alerts_ui_with_snow() -> None:
    render_alert_types_with_snow_ui(
        summary_with_snow_df,
        entities_df,
        snow_ticket_details_df,
    )


if widgets is None:
    print("ipywidgets not installed; defaulting to ServiceNow-enhanced view.")
    _render_alerts_ui_with_snow()
else:
    show_snow_toggle = widgets.Checkbox(
        value=True,
        description="Show ServiceNow ticket information",
        indent=False,
    )
    toggle_hint = widgets.HTML(
        value="<span style='font-size:12px;color:#475569'>Toggle on/off to switch ServiceNow ticket details in the UI.</span>"
    )
    ui_output = widgets.Output()

    def _render_alerts_ui(_: Any | None = None) -> None:
        with ui_output:
            clear_output(wait=True)
            if bool(show_snow_toggle.value):
                _render_alerts_ui_with_snow()
            else:
                _render_alerts_ui_without_snow()

    show_snow_toggle.observe(_render_alerts_ui, names="value")
    display(widgets.VBox([show_snow_toggle, toggle_hint, ui_output]))
    _render_alerts_ui()


In [ ]:
# Cell 4: Export Files

ts = RUN_TS.strftime("%Y%m%d_%H%M%S")
export_paths = export_all(
    summary_df=summary_with_snow_df,
    entities_df=entities_df,
    errors_df=errors_df,
    output_dir=str(RUN_DAY_DIR),
    ts=ts,
    snow_ticket_details_df=snow_ticket_details_df,
    export_xlsx=config["export_xlsx"],
    export_csv=config["export_csv"],
)

print("Export paths:")
for key, value in export_paths.items():
    print(f"- {key}: {value}")




